### <font color='red'>Please do not edit this file. </font> Go to <font color='blue'>*File > Save a copy in Drive*</font>.

# **OTH Course Knowledge Representation 2026**
## **Tutorial 1: Basic Knowledge Graph Infrastructure**

---

In this Tutorial, we will learn how to use RDFLib to work with RDF creation, serialization and visualization.

### How to use this extended working copy

- Run setup and demonstration cells in order.
- Complete the separate working code cells and the prediction/observation answer fields for each subpart. Open **Show solution** when you want to compare; solution code is displayed as text and does not run automatically.
- Exercises 1–11 share the RDF working graph. To restart them, rerun **Graph Creation** and continue in order. In particular, repeated creation of blank nodes adds new nodes.
- The RDFS and SHACL exercises provide fresh, independent starting graphs, so their outcomes do not depend on how you solved the RDF exercises.
- The original visualizations, RDF Bag, reification, graph editing, serialization and Protégé tasks are all included.

**Contents:** RDF basics and original visualizations → additional Graphviz/yFiles views → RDF Exercises 1–11 → Protégé → RDFS → SHACL.

**Direct entry to the new sections:** [RDFS preparation](#rdfs-entry) · [SHACL preparation](#shacl-entry). Each has its own minimal preparation cell; earlier visualization definitions enable the optional views.


## First Steps with RDFlib

**[RDFlib](https://github.com/RDFLib/rdflib)** is a Python library for creating, parsing, storing, and querying RDF data.  
In short: it gives you the building blocks to work with RDF graphs in Python.

### What RDFlib Provides

- **Parsers and serializers**
  - RDF/XML, N3, N-Triples, N-Quads, Turtle, TriX, JSON-LD, HexTuples, RDFa, Microdata
- **Store implementations**
  - in‑memory graphs
  - persistent on‑disk stores (e.g., BerkeleyDB)
  - remote SPARQL endpoints
- **Graph interface**
  - single graphs
  - multiple named graphs in a dataset
- **SPARQL 1.1**
  - supports both **queries** and **updates**


Some parsers, stores and external integrations require optional plugins. The exercises use RDFLib's Turtle and RDF/XML support.


We require the following packages:


*   **RDFlib** for working with RDF
*   **PyDotPlus**, **Graphviz** and **kglab** for visualization


Other Tools:

* https://rdfplayground.dcc.uchile.cl/
* https://sketch.zazuko.com/
Additional packages used in the extensions:

- **owlrl**: explicitly materialize RDFS consequences.
- **pyshacl**: validate a data graph against SHACL shapes.
- **yfiles_jupyter_graphs 2.x** and **ipywidgets**: interactive graph views.
- **pyvis**: the existing interactive view, also used through kglab.

The `graphviz` Python package also needs the **Graphviz system program `dot`**.
Check `dot -V` in a terminal. For a local installation, use the [Graphviz installation instructions](https://graphviz.org/download/); on macOS, for example, `brew install graphviz`.
In Colab, if necessary, install the system package with `!apt-get -qq install graphviz` in a separate cell.

The installation cell uses `%pip` so that packages are installed for the active notebook kernel.


In [ ]:
%pip install "rdflib>=7.1,<8" "owlrl>=7.1,<8" "pyshacl>=0.30,<1" pydotplus graphviz kglab pyvis "yfiles_jupyter_graphs>=2.0,<3" ipywidgets


In [ ]:
# Core RDFlib imports
import rdflib
from rdflib import Graph, Namespace, URIRef, BNode, Literal
from rdflib.namespace import RDF, FOAF, RDFS, XSD  # common built-in vocabularies

# Optional: graph visualization (requires Graphviz)
import io
import pydotplus
from IPython.display import display, Image
from rdflib.tools.rdf2dot import rdf2dot

from pathlib import Path
from IPython.display import HTML, SVG, FileLink
import shutil

DBR = Namespace("http://dbpedia.org/resource/")
DBO = Namespace("http://dbpedia.org/ontology/")
EX = Namespace("http://example.org/")

def copy_graph(source):
    """Copy triples and namespace bindings into a new in-memory graph."""
    result = Graph()
    for prefix, namespace in source.namespaces():
        result.bind(prefix, namespace)
    for triple in source:
        result.add(triple)
    return result

if shutil.which("dot") is None:
    print("Optional SVG/PNG rendering needs the Graphviz system program dot. RDF, RDFS and SHACL computations remain available.")


First, let's **create an RDF graph** about movies. The example is given in RDF Turtle serialization.
**Units matter:** this teaching graph uses `ex:runtimeMinutes` for integer minutes. DBpedia's `dbo:runtime` is defined in **seconds**, with range `xsd:double`; using 127 to mean minutes under that IRI would change its meaning. See the [DBpedia definition](https://dbpedia.org/ontology/runtime). The film graph is a small, locally prepared teaching dataset, not a live DBpedia download.


In [ ]:
# we create a graph from turtle data
from rdflib import Graph

ttl_data = """
PREFIX dbr: <http://dbpedia.org/resource/>
PREFIX dbo: <http://dbpedia.org/ontology/>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX ex: <http://example.org/>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>

# Actors
dbr:Tobey_Maguire        rdf:type  dbo:Actor ;
    dbo:awards           dbr:Saturn_Award ;
    ex:portrays          dbr:Peter_Parker .
dbr:Kirsten_Dunst        rdf:type  dbo:Actor ;
    dbo:awards           dbr:MTV_Movie_Award ;
    ex:portrays          dbr:Mary_Jane_Watson .
dbr:Christian_Bale       rdf:type  dbo:Actor ;
    dbo:awards           dbr:Academy_Award_for_Best_Actor ;
    ex:portrays          dbr:Bruce_Wayne .
dbr:Michael_Caine        rdf:type  dbo:Actor .
dbr:Matthew_McConaughey  rdf:type  dbo:Actor .
dbr:Anne_Hathaway        rdf:type  dbo:Actor .

# Directors
dbr:Sam_Raimi            rdf:type  dbo:Director ;
    dbo:awards           dbr:Saturn_Award .
dbr:Christopher_Nolan    rdf:type  dbo:Director ;
    dbo:awards           dbr:BAFTA_Award_for_Best_Direction .

# Producers
dbr:Laura_Ziskin         rdf:type  dbo:Producer .
dbr:Emma_Thomas          rdf:type  dbo:Producer .

# Films
dbr:Spider-Man_2  rdf:type  dbo:Film ;
    rdfs:label    "Spider-Man 2"@en , "Spider-Man 2"@de ;
    dbo:starring  dbr:Tobey_Maguire , dbr:Kirsten_Dunst ;
    dbo:director  dbr:Sam_Raimi ;
    dbo:producer  dbr:Laura_Ziskin ;
    ex:runtimeMinutes   "127"^^xsd:integer ;
    dbo:sequel    dbr:Spider-Man_3 ;
    ex:playsIn    dbr:New_York_City ;
    ex:hasTheme   dbr:Heroism .

dbr:Batman_Begins rdf:type  dbo:Film ;
    rdfs:label    "Batman Begins"@en , "Batman beginnt"@de ;
    dbo:starring  dbr:Christian_Bale , dbr:Michael_Caine ;
    dbo:director  dbr:Christopher_Nolan ;
    dbo:producer  dbr:Emma_Thomas ;
    ex:runtimeMinutes   "140"^^xsd:integer ;
    dbo:sequel    dbr:The_Dark_Knight ;
    ex:playsIn    dbr:Gotham_City ;
    ex:hasTheme   dbr:Vigilantism .

dbr:Interstellar rdf:type  dbo:Film ;
    rdfs:label    "Interstellar"@en , "Interstellar"@de ;
    dbo:starring  dbr:Matthew_McConaughey , dbr:Anne_Hathaway ;
    dbo:director  dbr:Christopher_Nolan ;
    dbo:producer  dbr:Emma_Thomas ;
    ex:runtimeMinutes   "169"^^xsd:integer ;
    ex:playsIn    dbr:Outer_Space ;
    ex:hasTheme   dbr:Survival .
"""

g = Graph()
g.parse(data=ttl_data, format="turtle")



In [ ]:
# print all triples
for s, p, o in g:
   print((s, p, o))

### RDF Graph Serialization

RDFlib can serialize the same graph in multiple formats (Turtle, RDF/XML, JSON‑LD, N‑Triples, …).  
Choose the format that fits your audience or downstream tools.


In [ ]:
# print graph in Turtle
g.serialize(destination="moviegraph01.ttl")

We can also serialise to different formats such as RDF XML

In [ ]:
print(g.serialize(format="xml"))    #print RDF/XML

In [ ]:
# save to file
g.serialize(destination="moviegraph01.rdf", format="xml")

Let's print out all the triples in our graph.

### Visualizing the Graph

[kglab](https://github.com/DerwenAI/kglab):
The kglab library provides a simple abstraction layer in Python 3.7+ for building knowledge graphs, leveraging Pandas, NetworkX, RAPIDS, RDFLib, Morph-KGC, pythonPSL, and many more.

In [ ]:
import kglab

In [ ]:
# lets import the graph
kg = kglab.KnowledgeGraph().load_rdf("moviegraph01.ttl")

In [ ]:
# Measure the graph: report total nodes and edges
measure = kglab.Measure()
measure.measure_graph(kg)

print(f"Nodes: {measure.get_node_count()}")
print(f"Edges: {measure.get_edge_count()}")


In [ ]:
# Style nodes by prefix: instances (dbr) vs classes (dbo)
VIS_STYLE = {
    "dbr": {"color": "grey", "size": 40},
    "dbo": {"color": "blue", "size": 50},
}

subgraph = kglab.SubgraphTensor(kg)
pyvis_graph = subgraph.build_pyvis_graph(notebook=True, style=VIS_STYLE)
# Embed JavaScript/CSS in HTML so no local lib/ folder is needed.
pyvis_graph.cdn_resources = "in_line"


In [ ]:
# Create the HTML document containing the original interactive visualization.
html_path = Path.cwd() / "moviegraph01.html"
pyvis_graph.force_atlas_2based()
pyvis_graph.show(str(html_path), notebook=True)

# Colab: download and open it in a browser. Local Jupyter: follow the file link.
try:
    from google.colab import files
except ImportError:
    display(FileLink(str(html_path)))
else:
    files.download(str(html_path))


## Render the graph in an IFrame

In [ ]:
from IPython.display import HTML

subgraph = kglab.SubgraphTensor(kg)
pyvis_graph = subgraph.build_pyvis_graph(notebook=True, style=VIS_STYLE)
pyvis_graph.cdn_resources = "in_line"

html_path = Path.cwd() / "kg.html"
pyvis_graph.save_graph(str(html_path))
display(HTML(html_path.read_text(encoding="utf-8")))


Here is another graph visualization that simply creates a png.
This visualization is more static, but also shows the URIs for all resources.


In [ ]:
def visualize(g):
    """Render an RDFLib graph as a static PNG (Graphviz required)."""
    import shutil
    if shutil.which("dot") is None:
        print("Optional Graphviz view unavailable: install dot to render it. Inspect the textual results below.")
        return
    stream = io.StringIO()
    rdf2dot(g, stream)
    dot = stream.getvalue()

    dg = pydotplus.graph_from_dot_data(dot)
    png = dg.create_png()
    display(Image(png))

In [ ]:
import graphviz

def visualize_fixed(g):
    """Render an RDFLib graph as an SVG using graphviz directly to avoid pydotplus bugs."""
    import shutil
    if shutil.which("dot") is None:
        print("Optional Graphviz view unavailable: install dot to render it. Inspect the textual results below.")
        return
    stream = io.StringIO()
    rdf2dot(g, stream)
    dot_code = stream.getvalue()

    # Verwende graphviz.Source direkt, um den DOT-Code zu rendern
    src = graphviz.Source(dot_code)
    display(src)

visualize_fixed(g)

And finally a version with clickable links.

In [ ]:
from IPython.display import SVG, display
import graphviz

def visualize_svg_with_links(g):
    """Render RDFLib graph as SVG with clickable URIs (using graphviz directly to avoid pydotplus bugs)."""
    import shutil
    if shutil.which("dot") is None:
        print("Optional Graphviz view unavailable: install dot to render it. Inspect the textual results below.")
        return
    stream = io.StringIO()
    rdf2dot(g, stream)
    dot_code = stream.getvalue()

    # graphviz.Source rendert den DOT-Code direkt zu SVG.
    # rdf2dot fügt automatisch das 'URL'-Attribut für URIs hinzu.
    src = graphviz.Source(dot_code)
    svg_data = src.pipe(format='svg').decode('utf-8')
    display(SVG(svg_data))

visualize_svg_with_links(g)

The original `rdf2dot` examples above remain available. They present literal properties as compact table fields and use labels as node captions. The next two views additionally make **every RDF triple visible as a labelled, directed edge**, including literal objects.


## Additional RDF views: Graphviz SVG and yFiles

The previous visualization examples remain available. Here we add two views of the same RDF triples:

- **Graphviz SVG** gives a stable, labelled diagram for explaining and discussing a small graph.
- **yFiles** lets us zoom, select items, inspect their data, and change layouts interactively.

Our adapters keep **one directed, labelled edge per triple**, including triples whose objects are literals. An IRI, a blank node, and a literal are different kinds of RDF terms. Literal language tags and datatypes remain visible. Different predicates between the same two terms remain separate edges.

**Legend:** blue ellipse = IRI; purple hexagon = blank node; yellow box = literal. Original edges are grey. When a reference graph is supplied, additional edges are orange and dashed. Focus nodes, for example from a SHACL report, are highlighted in red.

These helpers only prepare the view; they do not add or remove triples in the supplied RDF graph.


In [ ]:
import json
import graphviz
from IPython.display import SVG, display


def _rdf_term_key(term):
    """A typed identity; str(term) alone would lose RDF distinctions."""
    if isinstance(term, Literal):
        return (
            "literal", str(term),
            str(term.datatype) if term.datatype else "",
            term.language.lower() if term.language else "",
        )
    if isinstance(term, BNode):
        return ("bnode", str(term))
    if isinstance(term, URIRef):
        return ("iri", str(term))
    raise TypeError(f"Unsupported RDF term: {type(term).__name__}")


def _rdf_short_iri(graph, iri):
    """Use existing namespace bindings; do not invent bindings."""
    return graph.namespace_manager.normalizeUri(iri)


def _rdf_term_label(graph, term):
    if isinstance(term, URIRef):
        return _rdf_short_iri(graph, term)
    if isinstance(term, BNode):
        return f"_:{term}"
    label = json.dumps(str(term), ensure_ascii=False)
    if term.language:
        return f"{label}@{term.language}"
    if term.datatype:
        return f"{label}^^{_rdf_short_iri(graph, term.datatype)}"
    return label


def _rdf_term_properties(graph, term):
    """Plain values suitable for both tooltips and the yFiles data panel."""
    if isinstance(term, Literal):
        kind = "Literal"
    elif isinstance(term, BNode):
        kind = "Blank node"
    else:
        kind = "IRI"
    return {
        "label": _rdf_term_label(graph, term),
        "kind": kind,
        "value": str(term),
        "datatype": str(term.datatype) if isinstance(term, Literal) and term.datatype else "",
        "language": term.language or "" if isinstance(term, Literal) else "",
    }


def _rdf_view_data(graph):
    """Stable local IDs prevent label/IRI collisions and DOT port parsing."""
    terms = {term for s, p, o in graph for term in (s, o)}
    ordered_terms = sorted(terms, key=_rdf_term_key)
    ids = {_rdf_term_key(term): f"n{i}" for i, term in enumerate(ordered_terms)}
    triples = sorted(
        graph,
        key=lambda triple: tuple(_rdf_term_key(term) for term in triple),
    )
    return ordered_terms, ids, triples


### Graphviz: every RDF triple is visible

The Python package `graphviz` needs the Graphviz system program `dot` as well. The setup section checks this dependency. SVG stays sharp when zooming and can be saved for slides or the solution discussion.

Unlike the earlier compact `rdf2dot` rendering, this view draws literal objects and their predicate edges explicitly. IRI nodes and predicate edges link to their full IRIs.

The implementation is supplied: you can call `show_rdf_graphviz(your_graph)` without editing the helper.

[Graphviz Python documentation](https://graphviz.readthedocs.io/en/stable/manual.html#jupyter-notebooks)


In [ ]:
def rdf_to_graphviz(
    graph, asserted_graph=None, focus_nodes=None, title="RDF graph"
):
    """Build a labelled SVG diagram without changing the RDF input."""
    terms, ids, triples = _rdf_view_data(graph)
    asserted = set(asserted_graph) if asserted_graph is not None else None
    focus = {_rdf_term_key(term) for term in (focus_nodes or [])}

    legend = (
        "Blue ellipse: IRI | Purple hexagon: blank node | Yellow box: literal"
        "\nGrey solid: original | Orange dashed: additional | Red: focus node"
    )
    dot = graphviz.Digraph(
        "rdf_view",
        format="svg",
        strict=False,  # Preserve different predicates between the same nodes.
        graph_attr={
            "rankdir": "LR", "bgcolor": "white",
            "label": graphviz.escape(f"{title}\n{legend}"),
            "labelloc": "t", "fontsize": "11",
            "fontname": "Arial", "pad": "0.25", "nodesep": "0.35",
        },
        node_attr={"fontname": "Arial", "fontsize": "11", "margin": "0.12,0.08"},
        edge_attr={"fontname": "Arial", "fontsize": "10", "arrowsize": "0.7"},
    )

    for term in terms:
        key = _rdf_term_key(term)
        properties = _rdf_term_properties(graph, term)
        attributes = {
            "style": "filled", "color": "#64748b", "fontcolor": "#0f172a",
            "tooltip": graphviz.escape(
                "\n".join(f"{k}: {v}" for k, v in properties.items() if v)
            ),
        }
        if isinstance(term, Literal):
            attributes.update(shape="box", fillcolor="#fef3c7")
        elif isinstance(term, BNode):
            attributes.update(shape="hexagon", fillcolor="#ede9fe", style="filled,dashed")
        else:
            attributes.update(
                shape="ellipse", fillcolor="#dbeafe",
                URL=str(term), target="_blank",
            )
        if key in focus:
            attributes.update(color="#dc2626", fillcolor="#fee2e2", penwidth="2.5")
        dot.node(ids[key], label=graphviz.escape(properties["label"]), **attributes)

    for s, p, o in triples:
        additional = asserted is not None and (s, p, o) not in asserted
        color = "#d97706" if additional else "#475569"
        dot.edge(
            ids[_rdf_term_key(s)], ids[_rdf_term_key(o)],
            label=graphviz.escape(_rdf_short_iri(graph, p)),
            color=color, fontcolor=color,
            style="dashed" if additional else "solid",
            penwidth="1.7" if additional else "1.1",
            tooltip=graphviz.escape(str(p)), URL=str(p), target="_blank",
        )
    return dot


def show_rdf_graphviz(
    graph, asserted_graph=None, focus_nodes=None, title="RDF graph"
):
    """Render and display SVG in the notebook; return no duplicate display."""
    import shutil
    if shutil.which("dot") is None:
        print("Optional Graphviz view unavailable: install dot to render it. Inspect the textual results below.")
        return
    diagram = rdf_to_graphviz(
        graph, asserted_graph=asserted_graph,
        focus_nodes=focus_nodes, title=title,
    )
    display(SVG(diagram.pipe(format="svg")))


In [ ]:
# A small view is easier to discuss than the complete movie graph.
# The source graph g is left intact.
movie_svg_view = Graph()
for prefix, namespace in g.namespaces():
    movie_svg_view.bind(prefix, namespace)

visible_predicates = {
    RDF.type, RDFS.label, DBO.director, DBO.starring, EX.runtimeMinutes
}
for triple in g.triples((DBR["Spider-Man_2"], None, None)):
    if triple[1] in visible_predicates:
        movie_svg_view.add(triple)

show_rdf_graphviz(movie_svg_view, title="Spider-Man 2: resources and literal values")

# To export the same SVG later:
# rdf_to_graphviz(movie_svg_view).render("spider_man_2", cleanup=True)


### yFiles: explore the graph interactively

The following view uses [yFiles Graphs for Jupyter](https://www.yworks.com/products/yfiles-graphs-for-jupyter). Use the toolbar to change the layout, search for a node, and inspect selected nodes and edges in the **Data** panel.

- **Hierarchical** layout is useful for directed relations and class hierarchies.
- **Organic** layout is useful for exploring a more interconnected graph.
- The data panel contains full IRIs and literal metadata as well as the shorter labels.

The widget is free to use under its own [license](https://github.com/yWorks/yfiles-jupyter-graphs/blob/main/LICENSE.md). Jupyter, local notebooks, and Colab are supported. For an institution-hosted JupyterHub with an unfamiliar domain, follow the widget's [activation instructions](https://yworks.github.io/yfiles-jupyter-graphs/04_key_activation/) if an activation message appears.

The next setup cell enables custom widgets when running in Colab. Local Jupyter skips this Colab-specific step.


In [ ]:
try:
    from google.colab import output as colab_output
except ImportError:
    colab_output = None

if colab_output is not None:
    colab_output.enable_custom_widget_manager()


The supplied adapter builds the node and edge lists explicitly. This keeps labels, datatypes, language tags, directions, and parallel predicate edges under our control.

`rdf_to_yfiles(...)` returns a widget; display it with `display(...)`. The `asserted_graph` and `focus_nodes` arguments have the same meaning as in the SVG helper. Both helpers use the complete graph you pass to them; choose a smaller graph beforehand when you want a focused explanation.

[yFiles widget API and styling documentation](https://yworks.github.io/yfiles-jupyter-graphs/02_graph_widget/)


In [ ]:
def rdf_to_yfiles(
    graph, asserted_graph=None, focus_nodes=None, layout="hierarchical"
):
    """Build an interactive RDF view using yFiles Graphs for Jupyter 2.x."""
    from yfiles_jupyter_graphs import (
        GraphWidget, NodeStyle, NodeShape, EdgeStyle, DashStyle
    )

    terms, ids, triples = _rdf_view_data(graph)
    asserted = set(asserted_graph) if asserted_graph is not None else None
    focus = {_rdf_term_key(term) for term in (focus_nodes or [])}

    nodes = []
    for term in terms:
        properties = _rdf_term_properties(graph, term)
        properties["focus"] = _rdf_term_key(term) in focus
        nodes.append({"id": ids[_rdf_term_key(term)], "properties": properties})

    edges = []
    for index, (s, p, o) in enumerate(triples):
        additional = asserted is not None and (s, p, o) not in asserted
        edges.append({
            "id": f"e{index}",
            "start": ids[_rdf_term_key(s)],
            "end": ids[_rdf_term_key(o)],
            "properties": {
                "label": _rdf_short_iri(graph, p),
                "predicate": str(p),
                "statement": "additional" if additional else "original",
                "additional": additional,
            },
        })

    widget = GraphWidget(nodes=nodes, edges=edges, directed=True)
    widget.node_label_mapping = "label"
    widget.edge_label_mapping = "label"

    def node_style(node):
        properties = node["properties"]
        colors = {
            "IRI": "#dbeafe",
            "Blank node": "#ede9fe",
            "Literal": "#fef3c7",
        }
        shapes = {
            "IRI": NodeShape.ELLIPSE,
            "Blank node": NodeShape.HEXAGON,
            "Literal": NodeShape.RECTANGLE,
        }
        return NodeStyle(
            color="#fee2e2" if properties["focus"] else colors[properties["kind"]],
            shape=shapes[properties["kind"]],
        )

    def edge_style(edge):
        additional = edge["properties"]["additional"]
        return EdgeStyle(
            color="#d97706" if additional else "#475569",
            directed=True,
            thickness=2.0 if additional else 1.0,
            dash_style=DashStyle.DASH if additional else DashStyle.SOLID,
        )

    widget.node_styles_mapping = node_style
    widget.edge_styles_mapping = edge_style
    widget.graph_layout = layout
    widget.sidebar = {"enabled": True, "start_with": "Data"}
    widget.overview = True
    return widget


In [ ]:
# Explore all triples of the original movie graph.
# Click a literal node to inspect its datatype or language.
import importlib.util
if importlib.util.find_spec("yfiles_jupyter_graphs") is None:
    print("Optional yFiles view unavailable: install yfiles_jupyter_graphs and rerun this cell.")
    yfiles_movie_view = None
else:
    yfiles_movie_view = rdf_to_yfiles(g, layout="organic")
    display(yfiles_movie_view)

    # A second layout can be selected in the widget toolbar, or from Python:
    # yfiles_movie_view.graph_layout = "hierarchical"


### Reading the views carefully

Short labels are presentation choices: a node's identity still includes its RDF term kind and, for literals, its datatype or language. For example, a string `"152"` and an integer `"152"^^xsd:integer` are distinct RDF terms. So are `"Spider-Man 2"@en` and `"Spider-Man 2"@de`.

In the later RDFS examples, we will pass the original graph as `asserted_graph` to highlight new consequences. In the SHACL examples, `focus_nodes` can highlight resources mentioned in the validation report. A small view of the relevant triples is especially useful for the solution discussion.


## Graph Creation


*We* can look up information for a given URI.  
For example, let’s print all the facts about **Christopher Nolan**.

In [ ]:
chris = URIRef('http://dbpedia.org/resource/Christopher_Nolan')
# Print all predicate and objects of triples that have Christopher Nolan as subject
print([o for o in g.predicate_objects(subject=chris)])
print([o for o in g.subject_predicates(object=chris)])

In [ ]:
from rdflib import Namespace

DBR = Namespace("http://dbpedia.org/resource/")
person = DBR.Christian_Bale

for s, p, o in g.triples((person, None, None)):
    print(p, o)


Let's create a new simple graph and populate it with some information.

In [ ]:
DBR = Namespace("http://dbpedia.org/resource/")
DBO = Namespace("http://dbpedia.org/ontology/")
EX  = Namespace("http://example.org/")

In [ ]:
# create new graph instance g1 and bind to namespaces
g1 = Graph()
g1.bind("dbr", DBR)
g1.bind("dbo", DBO)
g1.bind("ex", EX)

# exactly 5 URI nodes: film, class, director, two actors
g1.add((DBR.The_Dark_Knight, DBO.director, DBR.Christopher_Nolan))
g1.add((DBR.The_Dark_Knight, DBO.starring, DBR.Christian_Bale))
g1.add((DBR.The_Dark_Knight, DBO.starring, DBR.Heath_Ledger))
g1.add((DBR.The_Dark_Knight, RDF.type, DBO.Film))

visualize_svg_with_links(g1)

### Working on the original RDF exercises

The complete eleven exercises follow. Each has a separate working cell and a closed, expandable reference solution. Copy solution code into a working cell only when you want to execute it. The original `g1` and later `mergedGraph` build on one another; rerun the initialization above to start over.

**Self-checks:** inspect the triples you intended to add, their literal datatypes and languages, and the effect of each edit. RDF graphs are sets: adding the same triple twice does not create a duplicate.


## Exercise 1 — Enrich the Dark Knight Graph

**Goal:** Add more facts to `g1` and re‑visualize.
Look them up from http://dbpedia.org/resource/The_Dark_Knight

**Pick any 3–5 proposals below and implement them:**
- Add `dbo:producer` (e.g., `dbr:Emma_Thomas`)
- Add `dbo:releaseDate` with an `xsd:date`
- Add `ex:runtimeMinutes` with an `xsd:integer`
- Add `dbo:genre` (e.g., `dbr:Crime_film`)
- Add `ex:hasTheme` (e.g., `dbr:Vigilantism`)
- Add `ex:playsIn` (e.g., `dbr:Gotham_City`)
- Add `ex:portrays` for actors (e.g., `dbr:Christian_Bale ex:portrays dbr:Bruce_Wayne`)

**Then:** run the visualization again.


In [ ]:
# Exercise 1 -- add your code here


<details>
<summary>Show solution — Exercise 1</summary>

```python
from rdflib.namespace import RDF, XSD

# Example additions (edit or extend)
g1.add((DBR.The_Dark_Knight, DBO.producer, DBR.Emma_Thomas))
g1.add((DBR.The_Dark_Knight, DBO.releaseDate, Literal("2008-07-18", datatype=XSD.date)))
g1.add((DBR.The_Dark_Knight, EX.runtimeMinutes, Literal(152, datatype=XSD.integer)))
g1.add((DBR.The_Dark_Knight, DBO.genre, DBR.Crime_film))
g1.add((DBR.The_Dark_Knight, EX.hasTheme, DBR.Vigilantism))
g1.add((DBR.The_Dark_Knight, EX.playsIn, DBR.Gotham_City))
g1.add((DBR.Christian_Bale, EX.portrays, DBR.Bruce_Wayne))

visualize_svg_with_links(g1)
```

The runtime is measured in minutes using our explicitly named local property. The added triples describe both the film and its actors.

</details>


## Exercise 2 — Model the Cast as an RDF Bag

**Goal:** Represent the cast as an unordered container.

**Tasks:**
1. Create a `rdf:Bag` container for the cast.
2. Link the film to the container using `ex:cast`.
3. Add at least three actors as `rdf:_1`, `rdf:_2`, `rdf:_3`.
4. Visualize the updated graph.

**Hint (RDFlib):** You may need a blank node. You can create a blank node with `BNode()` and use it like any other node.


In [ ]:
# Exercise 2 -- add your code here


<details>
<summary>Show solution — Exercise 2</summary>

```python
from rdflib import BNode
from rdflib.namespace import RDF

cast = BNode()

g1.add((DBR.The_Dark_Knight, EX.cast, cast))
g1.add((cast, RDF.type, RDF.Bag))
g1.add((cast, RDF._1, DBR.Christian_Bale))
g1.add((cast, RDF._2, DBR.Heath_Ledger))
g1.add((cast, RDF._3, DBR.Michael_Caine))

visualize_svg_with_links(g1)
```

An rdf:Bag expresses an intended unordered container even though its membership predicates have numbers. Re-running BNode() creates another container; rerun the g1 initialization to restart the exercise.

</details>


## Exercise 3 — Reception + Critique (Reification)

**Task:** Formalize the following text as RDF, and add a reception score.

**Text to formalize:**
> The Village Voice wrote that the performance would have made Ledger a legend even if he had lived.

**Also add:**
- A reception score of **94** from **Rotten Tomatoes**.

**Hints (relations to use):**
- `EX.receptionScore` for the Rotten Tomatoes number
- `EX.receptionSource` for the source (Rotten Tomatoes)
- Reification: `rdf:subject`, `rdf:predicate`, `rdf:object`
- `EX.source` for who said it (The Village Voice)
- `EX.comment` for the quote text


In [ ]:
# Exercise 3 -- add your code here


<details>
<summary>Show solution — Exercise 3</summary>

```python
from rdflib import BNode, Literal
from rdflib.namespace import RDF, XSD

# Reception score
g1.add((DBR.The_Dark_Knight, EX.receptionScore, Literal(94, datatype=XSD.integer)))
g1.add((DBR.The_Dark_Knight, EX.receptionSource, DBR.Rotten_Tomatoes))

# Reified critique
stmt = BNode()
g1.add((stmt, RDF.type, RDF.Statement))
g1.add((stmt, RDF.subject, DBR.Heath_Ledger))
g1.add((stmt, RDF.predicate, EX.performance))
g1.add((stmt, RDF.object, DBR.Legendary_Performance))

g1.add((stmt, EX.source, DBR.The_Village_Voice))
g1.add((stmt, EX.comment, Literal(
    "The performance would have made Ledger a legend even if he had lived."
)))
```

This deliberately simplified assessment triple does not capture the full counterfactual meaning of the quotation. The quotation is preserved as a literal. Reifying a statement does not automatically assert its subject–predicate–object triple. The score 94 is a fixed illustrative input, not a claim about a current rating.

</details>


## Exercise 4 — Literals, Datatypes, and Language Tags

**Goal:** Practice modeling *data values* in RDF.

**Tasks:**
1. Add a release date for the film as an `xsd:date`.
2. Add a runtime as an `xsd:integer`.
3. Add multilingual labels for the film (`@en` and `@de`).

**Hints (relations to use):**
- `dbo:releaseDate`
- `ex:runtimeMinutes`
- `rdfs:label`


In [ ]:
# Exercise 4 -- add your code here


<details>
<summary>Show solution — Exercise 4</summary>

```python
from rdflib import Literal
from rdflib.namespace import XSD, RDFS

g1.add((DBR.The_Dark_Knight, DBO.releaseDate, Literal("2008-07-18", datatype=XSD.date)))
g1.add((DBR.The_Dark_Knight, EX.runtimeMinutes, Literal(152, datatype=XSD.integer)))

g1.add((DBR.The_Dark_Knight, RDFS.label, Literal("The Dark Knight", lang="en")))
g1.add((DBR.The_Dark_Knight, RDFS.label, Literal("The Dark Knight", lang="de")))
```

Dates and runtime values have explicit datatypes. Language-tagged labels use rdf:langString; the English and German labels are different RDF terms even when their text is identical.

</details>


## Exercise 5 — Merge Two Graphs

**Goal:** Combine `g` and `g1` into a single graph mergedGraph.

**Tasks:**
1. Create a new graph `mergedGraph`.
2. Add all triples from `g` and `g1` into `mergedGraph`.
3. Print the triple count and visualize `mergedGraph`.

**Hints (Python functions):**
- Create a graph with `Graph()`
- Merge graphs with `mergedGraph += g` and `mergedGraph += g1`
- Count triples with `len(mergedGraph)`
- Bind the mergedGraph to the namespaces


In [ ]:
# Exercise 5 -- add your code here


<details>
<summary>Show solution — Exercise 5</summary>

```python
from rdflib import Graph

mergedGraph = Graph()


mergedGraph.bind("dbo", DBO)
mergedGraph.bind("dbr", DBR)
mergedGraph.bind("ex", EX)
mergedGraph.bind("rdf", RDF)
mergedGraph.bind("rdfs", RDFS)

mergedGraph += g
mergedGraph += g1

print(f"Triples in mergedGraph: {len(mergedGraph)}")
visualize_svg_with_links(mergedGraph)
```

The union contains the triples of both source graphs. Identical triples occur only once. The source graphs are not edited by adding their triples to a new graph.

</details>


## Exercise 6 — Add a New Film to the Merged Graph

**Goal:** Enrich the merged graph with a new film and multilingual labels.

**Tasks:**
1. Create a URI for `dbr:Die_Hard`.
2. Add an English label (`@en`) and two other languages (`@de`, `@uk`).
3. Add a release date as an `xsd:date`.
4. Visualize the updated `mergedGraph`.

**Hints (functions & relations):**
- Use `Literal("text", lang="en")` for language tags
- Use `Literal("YYYY-MM-DD", datatype=XSD.date)` for dates
- Use `RDFS.label` and `DBO.releaseDate`
- Add triples with `mergedGraph.add((s, p, o))`

Also type the new resource as `dbo:Film` so it can be selected as a film independently of any later inference.


In [ ]:
# Exercise 6 -- add your code here


<details>
<summary>Show solution — Exercise 6</summary>

```python
from rdflib import Literal, Namespace
from rdflib.namespace import RDFS, XSD

DBR = Namespace("http://dbpedia.org/resource/")
DBO = Namespace("http://dbpedia.org/ontology/")

diehard = DBR.Die_Hard

title_en = Literal("Die Hard", lang="en")
release_date = Literal("1988-07-15", datatype=XSD.date)

mergedGraph.add((diehard, RDF.type, DBO.Film))
mergedGraph.add((diehard, RDFS.label, title_en))
mergedGraph.add((diehard, DBO.releaseDate, release_date))

mergedGraph.add((diehard, RDFS.label, Literal("Stirb langsam", lang="de")))
mergedGraph.add((diehard, RDFS.label, Literal("Міцний горішок", lang="uk")))

visualize_svg_with_links(mergedGraph)
```

The new resource is explicitly typed as dbo:Film, which also makes it discoverable by a later sh:targetClass dbo:Film shape. The language code for Ukrainian is uk.

</details>


## Exercise 7 — Add vs. Set (Multiple vs. Single Values)

**Goal:** Understand the difference between `add()` and `set()`.

**Tasks:**
1. Add two different release dates for `dbr:Die_Hard` using `add(...)`.
2. Print all release dates (you should see both).
3. Use `set(...)` to keep only **one** release date.
4. Print again and verify only one remains.

**Hints (functions):**
- `mergedGraph.add((s, p, o))` keeps existing values
- `mergedGraph.set((s, p, o))` replaces existing values
- Use `mergedGraph.objects(subject=diehard, predicate=DBO.releaseDate)` to list values


In [ ]:
# Exercise 7 -- add your code here


<details>
<summary>Show solution — Exercise 7</summary>

```python
from rdflib import Literal
from rdflib.namespace import XSD

# Step 1: add two dates
mergedGraph.add((diehard, DBO.releaseDate, Literal("1988-07-15", datatype=XSD.date)))
mergedGraph.add((diehard, DBO.releaseDate, Literal("1988-07-22", datatype=XSD.date)))

print("After add:")
for date in mergedGraph.objects(subject=diehard, predicate=DBO.releaseDate):
    print(date)

# Step 3: replace with one date
mergedGraph.set((diehard, DBO.releaseDate, Literal("1988-07-22", datatype=XSD.date)))

print("After set:")
for date in mergedGraph.objects(subject=diehard, predicate=DBO.releaseDate):
    print(date)
```

add() preserves other values. set() replaces existing objects for this particular subject and predicate. It does not declare a global cardinality constraint.

</details>


## Exercise 8 — Migrate a Predicate (Schema Change)

Suppose you changed your schema (`releaseDate → releaseDateUSA`) and want the data to match.

**Tasks:**
1. Find all triples with `DBO.releaseDate`.
2. Replace them with `EX.releaseDateUSA`.
3. Verify that no `DBO.releaseDate` triples remain.

**Hints (functions):**
- Use `mergedGraph.triples((None, DBO.releaseDate, None))`
- Use `mergedGraph.add(...)` and `mergedGraph.remove(...)`
- Use `list(...)` to avoid modifying while iterating

**Modelling assumption:** the dates in this exercise refer to the selected US release. Changing the property also narrows the meaning.


In [ ]:
# Exercise 8 -- add your code here


<details>
<summary>Show solution — Exercise 8</summary>

```python
# migrate predicate DBO.releaseDate -> EX.releaseDateUSA
for s, p, o in list(mergedGraph.triples((None, DBO.releaseDate, None))):
    mergedGraph.remove((s, DBO.releaseDate, o))
    mergedGraph.add((s, EX.releaseDateUSA, o))

# check: should be 0
print(len(list(mergedGraph.triples((None, DBO.releaseDate, None)))))
```

This migration is justified only when the original dates really refer to the selected US release. A general release date cannot be relabelled as a US date without that knowledge. RDFS Exercise R3 later explores a subproperty relationship instead.

</details>


## Exercise 9 — Write a Function to Print Film Facts

**Goal:** Write a function that prints all predicate–object pairs for a film.

**Tasks:**
1. Write `print_film_facts(graph, film_uri)`.
2. Call it for **The Dark Knight** and **Spider‑Man 2**.

**Hints (functions):**
- Use `graph.predicate_objects(subject=film_uri)`
- Print each predicate–object pair on its own line


In [ ]:
# Exercise 9 -- add your code here


<details>
<summary>Show solution — Exercise 9</summary>

```python
def print_film_facts(graph, film_uri):
    for p, o in graph.predicate_objects(subject=film_uri):
        print(p, o)

print("The Dark Knight")
print_film_facts(mergedGraph, DBR.The_Dark_Knight)

print("\nSpider-Man 2")
print_film_facts(mergedGraph, DBR["Spider-Man_2"])
```

predicate_objects() selects outgoing triples for the requested subject; it does not perform RDFS inference by itself.

</details>


## Exercise 10 — Undo a Change (Remove a Film)

Suppose you want to undo a change like adding **Die Hard**.  
Write a function that removes all outgoing triples about a film and confirms they are gone.

**Tasks:**
1. Write `remove_film(graph, film_uri)` to delete all triples where the film is the subject.
2. Verify that no triples remain for that subject.

**Hints (functions):**
- Use `graph.remove((film_uri, None, None))`
- Check with a loop or `len(list(graph.triples((film_uri, None, None))))`


In [ ]:
# Exercise 10 -- add your code here


<details>
<summary>Show solution — Exercise 10</summary>

```python
def remove_film(graph, film_uri):
    graph.remove((film_uri, None, None))

def has_film_triples(graph, film_uri):
    return len(list(graph.triples((film_uri, None, None)))) > 0

remove_film(mergedGraph, diehard)
print("Still present?", has_film_triples(mergedGraph, diehard))
```

This removes outgoing triples only. Incoming references to the film and disconnected auxiliary blank nodes may remain. The check therefore confirms absence as a subject, not absence from the whole graph.

</details>


## Exercise 11 — Serialize and Open in Protégé

**Goal:** Export the merged graph to RDF/XML and inspect it in Protégé.

**Tasks:**
1. Serialize `mergedGraph` to RDF/XML and save it as `mergedGraph.rdf`.
2. Open the file in Protégé and inspect the classes and individuals.

**Hints (functions):**
- Use `mergedGraph.serialize(destination="mergedGraph.rdf", format="xml")`


In [ ]:
# Exercise 11 -- add your code here


<details>
<summary>Show solution — Exercise 11</summary>

```python
mergedGraph.serialize(destination="mergedGraph.rdf", format="xml")
```

RDF/XML is a serialization of the same RDF graph. Protégé's presentation depends on its installed tabs, plugins and interpretation of RDF/OWL vocabulary.

</details>


## Protégé (Ontology Editor) -- Exercises

[Protégé](https://protege.stanford.edu/) is a free, widely used ontology editor for building and exploring RDF/OWL knowledge graphs.

**Please download and install Protégé** from the official website before the next session:  
https://protege.stanford.edu/

## Tasks

1. Open `mergedGraph.rdf`.  
   Can you find the individual `dbr:The_Dark_Knight`?

<details>
<summary>Show solution</summary>

In the **Individuals** tab, search for `The_Dark_Knight`.  
If prefixes are missing, it may show as a full URI or with `ns1:`.

</details>

2. What classes are visible in the **Classes** tab?  
   Which one is used for films?

<details>
<summary>Show solution</summary>

Look for `dbo:Film` (or `ns1:Film`).  
That is the class used for films.

</details>

3. In the **Individuals** tab, which actors appear?  
   Pick one and list all its properties.

<details>
<summary>Show solution</summary>

Example actors: `Tobey_Maguire`, `Christian_Bale`, `Heath_Ledger`.  
Click one to see properties like `rdf:type`, `ex:portrays`, `dbo:awards`.

</details>

4. Can you see multilingual labels (`rdfs:label`) for any film?  
   Which languages are shown?

<details>
<summary>Show solution</summary>

Example: `The Dark Knight` shows `@en` and `@de`.  
You’ll see language tags next to the label values.

</details>

5. Find the property `ex:runtimeMinutes`.  
   Which individuals have it, and what are the values?

<details>
<summary>Show solution</summary>

Films like `The_Dark_Knight`, `Interstellar`, and `Spider-Man_2` have it.  
Values are typed integers (e.g., 152, 169, 127).

</details>

6. Locate the blank node that represents the cast (`rdf:Bag`).  
   Which actors are contained?

<details>
<summary>Show solution</summary>

It appears as an anonymous node (blank node ID).  
Inside it you’ll see `rdf:_1`, `rdf:_2`, `rdf:_3` pointing to actors.

</details>

7. Check the `EX.receptionScore` of **The Dark Knight**.  
   What datatype is used?

<details>
<summary>Show solution</summary>

The value is `94` with datatype `xsd:integer`.

</details>

8. Add a new label in another language (e.g., `@es`) and save.  
   Does it appear in the RDF file?

<details>
<summary>Show solution</summary>

Yes. Open the exported file and confirm the new `rdfs:label` with `@es`.

</details>

9. Explore the Graph View in Protégé (OntoGraf)
<details> <summary>Show solution</summary>
Open the OntoGraf tab:
Window → Tabs → OntoGraf

Select a class or individual (e.g., dbo:Film) and click Add to the graph.

Expand neighbors to see related nodes and properties.

</details>

10. Enable the SPARQL tab and run your first query.

1. Go to `Window → Tabs → SPARQL` to enable the tab.  
2. Paste and run:

```sparql
PREFIX dbr: <http://dbpedia.org/resource/>
PREFIX dbo: <http://dbpedia.org/ontology/>

SELECT ?film ?director WHERE {
  ?film dbo:director ?director .
}
```


<details>
<summary>Show solution — Protégé task 10</summary>

Enable an available SPARQL query tab (its availability depends on installed plugins), run the query above, and inspect each film–director pair. After the reference RDF exercises, the graph includes The Dark Knight, Spider-Man 2, Batman Begins and Interstellar with their directors. Die Hard has been removed as a subject in Exercise 10.

If the SPARQL tab is unavailable in your Protégé installation, the same graph can also be queried in RDFLib:

```python
for film, director in mergedGraph.subject_objects(DBO.director):
    print(film, director)
```

</details>


<a id="rdfs-entry"></a>

# Part II — RDF Schema and inference with owlrl

RDF tells us how to state facts. RDFS gives vocabulary for class and property hierarchies and for drawing additional conclusions from those facts.

We keep using the film domain. In this part, the data and schema are small, explicitly supplied **teaching examples**. They are not a complete reproduction of DBpedia's ontology. Using a DBpedia IRI or binding a prefix does not download any external schema.

**Learning goals**

- Distinguish an instance's type from a relationship between classes.
- Predict conclusions from subclasses, subproperties, domain and range.
- Run RDFS inference explicitly with owlrl and identify which assertions are new.
- Explain why RDFS inference does not check our application's required fields.

All four exercises below are retained in full. The session has 60 minutes of working time and 30 minutes of discussion; any unfinished experiments can be continued afterwards.


### Start this section — provided preparation

**Run the next cell as supplied.** You can start here in a fresh kernel without executing any earlier exercise solutions. It imports RDF terms, defines the prefixes and supplies graph-copying and table helpers. The following section-specific helper cell supplies inference or validation functions.

The setup code is infrastructure and may appear collapsed in Jupyter. Its usage is explained in the local **How to…** folds. Complete only cells labelled **Your code / Student work**; use the separate answer fields for predictions and observations.

**Optional views:** SVG and yFiles examples are retained. To enable them, run the helper-definition cells under **Additional RDF views** earlier in this notebook. Core tables, inference and validation work without those renderers. Reset an experiment by rerunning its supplied preparation, then the required subparts in order.


In [ ]:
# PROVIDED PREPARATION — run as supplied. Safe to rerun.
# No earlier exercise solution or visualization package is required.
import importlib.util
import shutil
from html import escape as html_escape
from rdflib import Graph, Namespace, URIRef, BNode, Literal
from rdflib.namespace import RDF, RDFS, XSD
from IPython.display import HTML, display

DBR = Namespace("http://dbpedia.org/resource/")
DBO = Namespace("http://dbpedia.org/ontology/")
EX = Namespace("http://example.org/")

def copy_graph(source):
    """Copy triples and prefix bindings; leave the original graph unchanged."""
    result = Graph()
    for prefix, namespace in source.namespaces():
        result.bind(prefix, namespace)
    for triple in source:
        result.add(triple)
    return result

def display_rows(rows, columns=None, title=""):
    """Display supplied result records as a readable, escaped HTML table."""
    rows = list(rows)
    if title:
        display(HTML("<p><strong>" + html_escape(str(title)) + "</strong></p>"))
    if not rows:
        print("No rows.")
        return
    columns = list(columns) if columns is not None else list(rows[0])
    header = "<tr>" + "".join("<th>" + html_escape(str(c)) + "</th>" for c in columns) + "</tr>"
    body = []
    for row in rows:
        body.append("<tr>" + "".join("<td>" + html_escape(str(row.get(c, "—"))) + "</td>" for c in columns) + "</tr>")
    display(HTML("<table><thead>" + header + "</thead><tbody>" + "".join(body) + "</tbody></table>"))

def show_memberships(asserted_graph, inferred_graph, candidates, title="Selected type statements"):
    """Show actual graph membership before/after inference, not expected answers."""
    rows = []
    for triple in candidates:
        rows.append({
            "Statement (subject, predicate, object)": " ".join(t.n3(inferred_graph.namespace_manager) for t in triple),
            "Asserted": triple in asserted_graph,
            "After inference": triple in inferred_graph,
        })
    display_rows(rows, title=title)

def optional_svg(graph, asserted_graph=None, focus_nodes=None, title="RDF graph"):
    """Display an available SVG helper; optional views never supply evidence."""
    renderer = globals().get("show_rdf_graphviz")
    if renderer is None:
        print("Optional SVG view: run the earlier Additional RDF views helper-definition cells to enable it. The tables above contain the computation results.")
        return
    if shutil.which("dot") is None:
        print("Optional SVG view needs Graphviz dot. The tables above remain available.")
        return
    renderer(graph, asserted_graph=asserted_graph, focus_nodes=focus_nodes, title=title)

def optional_yfiles(graph, asserted_graph=None, focus_nodes=None, layout="hierarchical"):
    """Display an available yFiles adapter, preserving the textual learning path."""
    adapter = globals().get("rdf_to_yfiles")
    if adapter is None or importlib.util.find_spec("yfiles_jupyter_graphs") is None:
        print("Optional yFiles view: install its package and run the earlier Additional RDF views helper-definition cells to enable it.")
        return
    widget = adapter(graph, asserted_graph=asserted_graph, focus_nodes=focus_nodes, layout=layout)
    display(widget)

print("Core preparation ready: RDF terms, namespaces, graph copying and result tables.")
print("Supplied helpers: copy_graph, display_rows, show_memberships, optional_svg, optional_yfiles.")


### Your route through this section

Run the **Start this section** preparation supplied here, then the provided helper cell below. Supplied code is labelled **Provided**; exercise cells marked **Your code** are for your work. Each exercise has independent preparation, followed by smaller questions. Predict before running a new experiment and record an explanation after seeing its result.

We follow **R1 → R3 → R2 → combined example → R4**: class hierarchies, the analogous property hierarchy, domain/range reasoning, consolidation, then diagnosis. The original exercise identifiers are retained so earlier references still work.

Each subpart has local **How to…** guidance and its own closed solution. These helpers explain the mechanics without requiring you to open the answer. The complete original descriptions and detailed reference solutions remain available at the end of each exercise.

**Visible checkpoints:** use the small result tables to read what happened. Assertions supplement these observations: success is silent; failure raises an error. “Not in the inferred graph” means “not derived from these premises”, not “false in the world”.


## How to use owlrl

**RDFLib** stores and queries triples. Adding a schema triple to an RDFLib graph does not automatically materialize its consequences.

The Python package **owlrl** provides rule-based inference. Despite its name, it also offers a separate RDFS rule set. In this tutorial we select **RDFS_Semantics**. This does not introduce an OWL exercise.

The notebook's installation cell includes owlrl. If you work in a separate notebook, install it into that notebook's kernel with the following command:


```python
%pip install rdflib owlrl
```

The basic workflow is:

1. Create an asserted data graph and a schema graph.
2. Copy their triples into one working graph.
3. Call **DeductiveClosure(RDFS_Semantics).expand(working_graph)**.
4. Compare selected triples before and after the call.

**expand changes its graph in place.** It adds inferred triples to the supplied object; use the graph afterwards rather than assigning the call's return value. Rules run repeatedly until no more consequences can be added. A chain of several rules is handled by one call.

See the [OWL-RL package documentation](https://owl-rl.readthedocs.io/en/latest/owlrl.html) and the [RDFS vocabulary specification](https://www.w3.org/TR/rdf-schema/).


**Provided helper — run once.** `rdfs_closure(data, schema)` is our notebook helper. Inside it, the actual owlrl call is `DeductiveClosure(RDFS_Semantics).expand(...)`. It returns two graphs: the asserted union and a separately materialized copy.


In [ ]:
from owlrl import DeductiveClosure, RDFS_Semantics

def rdfs_closure(data_graph, schema_graph):
    """Keep inputs unchanged; return asserted union and materialized RDFS graph."""
    asserted_graph = copy_graph(data_graph)
    for prefix, namespace in schema_graph.namespaces():
        asserted_graph.bind(prefix, namespace)
    for triple in schema_graph:
        asserted_graph.add(triple)

    inferred_graph = copy_graph(asserted_graph)
    DeductiveClosure(RDFS_Semantics).expand(inferred_graph)
    return asserted_graph, inferred_graph


## Guided first experiment — one rule, one new type

**What is this about?** An Actor instance can also be recognized as a Person because a class relationship connects those types. We first learn what the library changes using only one rule.

**After this example, you can** identify the asserted graph and its working copy, and read one before/after membership check.

**Predict:** Does adding the subclass statement alone change a normal RDFLib membership lookup? What changes after the explicit owlrl call?

The next two cells are **provided demonstrations**, not exercise answers.


In [ ]:
intro_data=Graph()
intro_data.bind("dbr",DBR); intro_data.bind("dbo",DBO)
intro_data.add((DBR.Anne_Hathaway,RDF.type,DBO.Actor))
intro_schema=Graph()
intro_schema.add((DBO.Actor,RDFS.subClassOf,DBO.Person))
intro_asserted=copy_graph(intro_data)
for triple in intro_schema:
    intro_asserted.add(triple)
intro_candidate=(DBR.Anne_Hathaway,RDF.type,DBO.Person)
print("Before explicit inference:",intro_candidate in intro_asserted)


In [ ]:
intro_inferred=copy_graph(intro_asserted)
DeductiveClosure(RDFS_Semantics).expand(intro_inferred)
show_memberships(intro_asserted,intro_inferred,[intro_candidate],
                 title="One supplied class rule: inspect the before/after result")
assert intro_candidate not in intro_asserted and intro_candidate in intro_inferred


Read the table from left to right: the candidate statement, whether it occurs in the asserted input, and whether it occurs after expansion. The expected statuses are **False → True**. The rule combines Anne Hathaway's Actor membership with Actor-subclass-Person.

`expand` changes `intro_inferred` in place. Its return value is not the result graph; continue using the supplied working graph. `intro_asserted` remains unchanged. The next exercises use `rdfs_closure` to perform this copying and expansion consistently.


## Exercise R1 — How do class relationships give resources additional types?

**What is this about?** A consumer looking for Persons should also find resources described as Actors or Directors. You will model these relationships once, then investigate which instance types follow.

**After this exercise, you can:** predict inherited memberships along a two-step hierarchy; distinguish subclass statements from instance membership; explain why a general type does not establish a specific role.

**Given:** `r1_data` and `r1_schema_start` below. **Your result:** a schema, a before/after table, and a short explanation of the two limit checks.

**Reset:** rerun this preparation and R1b before repeating R1c–d. The data graph stays unchanged.


In [ ]:
r1_data = Graph().parse(data="""
@prefix dbr: <http://dbpedia.org/resource/> .
@prefix dbo: <http://dbpedia.org/ontology/> .
@prefix ex: <http://example.org/> .
dbr:Christian_Bale a dbo:Actor .
dbr:Heath_Ledger a dbo:Actor .
dbr:Christopher_Nolan a dbo:Director .
ex:KnownPerson a dbo:Person .
""", format="turtle")

r1_schema_start = Graph().parse(data="""
@prefix dbo: <http://dbpedia.org/ontology/> .
@prefix ex: <http://example.org/> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
dbo:Actor a rdfs:Class .
dbo:Director a rdfs:Class .
dbo:Person a rdfs:Class .
ex:Agent a rdfs:Class .
""", format="turtle")


### R1a — Predict before coding

Assume Actor → Person, Director → Person, and Person → Agent. Complete the predictions now; leave the observed column for R1c.

| Statement | My prediction after inference | Observed in R1c |
|---|---|---|
| Christian Bale is a Person | … | … |
| Christian Bale is an Agent | … | … |
| Christopher Nolan is a Person | … | … |
| Christopher Nolan is an Agent | … | … |

**Your explanation:** Which statements would form a two-step chain? …


<details>
<summary>Show prediction explanation — R1a</summary>

Both people acquire Person and then Agent memberships. Bale starts as Actor; Nolan starts as Director. The subclass statements connect each supplied type to Person, and Person to Agent. These are predictions from the intended hierarchy; R1c will check the computed graph.

</details>


### R1b — Build the class hierarchy

**Your code:** copy `r1_schema_start` into `r1_schema`, add the three subclass relationships, and display those relationships. Keep instance facts in `r1_data`.

**Checkpoint:** the output should be a small table of class-to-class relationships, with `rdfs:subClassOf` as predicate.


<details>
<summary>How to add a schema triple</summary>

`copy_graph(source)` is a supplied notebook helper that preserves triples and prefix bindings. RDFLib's `Graph.add` accepts one tuple of three RDF terms.

A neutral example is `schema.add((EX.Specialist, RDFS.subClassOf, EX.Person))`. This relates two classes. An instance statement instead uses `RDF.type`.

Use `schema.triples((None, RDFS.subClassOf, None))` to retrieve all subclass statements. `None` is a wildcard. `term.n3(schema.namespace_manager)` produces a readable prefixed name. `display_rows` is our supplied display helper; it is not part of RDFLib.

</details>


In [ ]:
# R1b — create r1_schema and show its subclass triples.
# Add your code here.


**Your observation — R1b**

Which three schema relationships did your table show? Are the subject and object classes or particular people?

**My observation:** …


<details>
<summary>Show solution — R1b</summary>

```python
r1_schema=copy_graph(r1_schema_start)
r1_schema.add((DBO.Actor,RDFS.subClassOf,DBO.Person))
r1_schema.add((DBO.Director,RDFS.subClassOf,DBO.Person))
r1_schema.add((DBO.Person,RDFS.subClassOf,EX.Agent))
display_rows([
    {"subclass":s.n3(r1_schema.namespace_manager),
     "predicate":p.n3(r1_schema.namespace_manager),
     "superclass":o.n3(r1_schema.namespace_manager)}
    for s,p,o in sorted(r1_schema.triples((None,RDFS.subClassOf,None)),key=str)
],title="R1b: the three relationships you supplied")
assert len(list(r1_schema.triples((None,RDFS.subClassOf,None))))==3
```

The rows are the schema statements you entered, not newly inferred instance types. Read each row as “every instance of the left class is also an instance of the right class”. The assertion silently checks that the intended three relationships were entered.

</details>


### R1c — Run inference and compare with your predictions

**Requires:** your `r1_schema` from R1b. **Your code:** compute `r1_asserted` and `r1_inferred`, then display the four type statements from R1a before and after inference.

**Your observation:** Which rows changed? Explain one change using the original membership and one or two subclass statements. …


<details>
<summary>How to run inference and inspect a membership</summary>

Call `asserted, inferred = rdfs_closure(data, schema)`. The first graph contains the supplied data and schema; the second additionally contains their consequences. The helper calls owlrl internally and keeps your input graphs unchanged.

`candidate in graph` evaluates to a Boolean. The supplied `show_memberships(asserted, inferred, candidates, title="...")` displays a list of candidate triples as before/after rows. Use a triple such as `(resource, RDF.type, class_iri)` to ask about a type. Choose candidates before executing, so checking them does not replace making a prediction.

</details>


In [ ]:
# R1c — compute closure and compare the four selected type statements.
# Add your code here.


**Your observation — R1c**

Complete the observed column of R1a. Select one newly present membership and explain its complete rule chain.

**My observation and justification:** …


<details>
<summary>Show solution — R1c</summary>

```python
r1_asserted,r1_inferred=rdfs_closure(r1_data,r1_schema)
r1_candidates=[
    (DBR.Christian_Bale,RDF.type,DBO.Person),
    (DBR.Christian_Bale,RDF.type,EX.Agent),
    (DBR.Christopher_Nolan,RDF.type,DBO.Person),
    (DBR.Christopher_Nolan,RDF.type,EX.Agent),
]
show_memberships(r1_asserted,r1_inferred,r1_candidates,
                 title="R1c: predicted memberships before and after inference")
assert all(t not in r1_asserted and t in r1_inferred for t in r1_candidates)
```

All four selected rows change from False to True. In the Actor case, Actor → Person supplies the first new type, then Person → Agent supplies the next. One expansion computes the whole chain. The source data still has its original explicit memberships; the new statements are in the materialized copy.

</details>


### R1d — Investigate the limits

**Requires:** R1c. Test whether `KnownPerson` is an Actor, whether the class resource `dbo:Actor` is itself a Person, and whether that resource is an `rdfs:Class`. Display the three results before writing your interpretation.

**Your explanation:** What is the difference between an absent conclusion and a statement that is false? Why is the class-versus-instance question different from the preceding person checks? …


<details>
<summary>How to read a positive or negative membership check</summary>

`triple in graph` is True when that exact triple is stored. `triple not in graph` is its Python membership negation; it does not add a logical negation to RDF.

`assert condition` checks a Boolean and prints nothing when it succeeds. If it fails, Python raises `AssertionError`. Display your observations first so that you can interpret the result even when a check fails.

</details>


In [ ]:
# R1d — display the two limit checks and the class declaration check.
# Add your code here.


**Your observation — R1d**

Record the three after-inference statuses. Explain the difference between a superclass, an instance, and an absent conclusion.

**My interpretation:** …


<details>
<summary>Show solution — R1d</summary>

```python
r1_limit_candidates=[
    (EX.KnownPerson,RDF.type,DBO.Actor),
    (DBO.Actor,RDF.type,DBO.Person),
    (DBO.Actor,RDF.type,RDFS.Class),
]
show_memberships(r1_asserted,r1_inferred,r1_limit_candidates,
                 title="R1d: specific role and class-resource checks")
assert r1_limit_candidates[0] not in r1_inferred
assert r1_limit_candidates[1] not in r1_inferred
assert r1_limit_candidates[2] in r1_inferred
```

The after-inference results are False, False, True. Person membership does not justify the more specific Actor role. A subclass relationship propagates memberships of instances; it does not make the Actor class resource a Person. The False checks establish absence from this closure, not real-world negative facts.

</details>


<details>
<summary>Original complete task description — R1</summary>

## Exercise R1 — Classes, instances and transitive inference

**Goal:** Build a class hierarchy and distinguish it from instance membership.

The preparation cell supplies two actors, one director and one resource known only to be a Person. It also declares the relevant classes without connecting them.

**Tasks**

1. Copy r1_schema_start into your own r1_schema.
2. Add the subclass relationships Actor → Person, Director → Person, and Person → Agent.
3. Before running inference, predict which new types Christian Bale and Christopher Nolan acquire.
4. Run rdfs_closure and check your predictions.
5. Does being a Person imply being an Actor? Does making Actor a subclass of Person make the Actor class itself an instance of Person?

**Hints:** Use RDFS.subClassOf for relationships between classes and RDF.type for instance membership. Test a triple with the Python expression triple in graph.

</details>


<details>
<summary>Complete reference solution — R1</summary>

Add the hierarchy to a fresh schema copy. Both people inherit the Agent type through Person. Neither the reverse instance-membership implication nor the claim that the Actor class itself is a Person follows from these premises.

```python
r1_schema = copy_graph(r1_schema_start)
r1_schema.add((DBO.Actor, RDFS.subClassOf, DBO.Person))
r1_schema.add((DBO.Director, RDFS.subClassOf, DBO.Person))
r1_schema.add((DBO.Person, RDFS.subClassOf, EX.Agent))
r1_asserted, r1_inferred = rdfs_closure(r1_data, r1_schema)

for person in (DBR.Christian_Bale, DBR.Christopher_Nolan):
    assert (person, RDF.type, DBO.Person) in r1_inferred
    assert (person, RDF.type, EX.Agent) in r1_inferred
    print(person.n3(r1_inferred.namespace_manager), "is a Person and an Agent")

assert (EX.KnownPerson, RDF.type, DBO.Actor) not in r1_inferred
assert (DBO.Actor, RDF.type, DBO.Person) not in r1_inferred
assert (DBO.Actor, RDF.type, RDFS.Class) in r1_inferred
```

The class hierarchy is transitive. Instance membership propagates from a more specific class to its superclasses. The converse need not hold: KnownPerson may or may not be an actor, but this graph gives no basis to conclude that it is one.

---

**How to read the code and its output**

`r1_schema` is the copied schema with the three new subclass relationships. `rdfs_closure(...)` returns the asserted data-and-schema union as `r1_asserted` and its expanded copy as `r1_inferred`.

The `for` loop checks Bale and Nolan separately. An expression such as `(person, RDF.type, DBO.Person) in r1_inferred` returns a Boolean. `assert` continues silently when that Boolean is `True`; when it is `False`, Python raises an `AssertionError` and stops the current cell. For each person, the code prints a line only after both membership checks succeed:

```text
dbr:Christian_Bale is a Person and an Agent
dbr:Christopher_Nolan is a Person and an Agent
```

The final three assertions have no print statements. They check the two absent memberships and the Actor resource's class membership. If the cell finishes without an error, these checks also passed.

The supplied data supports these additional memberships:

| Resource | Asserted type | New types relevant to this exercise |
|---|---|---|
| Christian Bale | Actor | Person, Agent |
| Heath Ledger | Actor | Person, Agent |
| Christopher Nolan | Director | Person, Agent |
| KnownPerson | Person | Agent |

The table includes Ledger and KnownPerson, although the printing loop visits only Bale and Nolan. For Bale, Actor membership combines with Actor-subclass-Person, and the resulting Person membership combines with Person-subclass-Agent. One `expand(...)` call follows the whole chain.

The separate check about `dbo:Actor` concerns the **class resource**, not one of its instances. Its instances inherit Person membership; the class resource is typed as `rdfs:Class`. Finally, `not in` establishes that the expanded example contains no such triple. It does not prove that KnownPerson is not an actor in the real world.

</details>


## Exercise R3 — How can a general relationship expose a more specific fact?

**What is this about?** A US release date is a particular kind of release date. An application querying the general relationship should be able to use that more specific fact while its original meaning remains available.

**After this exercise, you can:** predict the direction of a subproperty inference; distinguish an added consequence from a destructive predicate migration.

**Given:** two supplied date facts and `r3_schema_start`. **Your result:** a rule, a four-row comparison, a diagram when available, and a brief explanation of the missing reverse inference.

We keep the identifier R3 and place it here because property hierarchies are analogous to the class hierarchies just studied. **Reset:** rerun the preparation and R3b before R3c.


In [ ]:
r3_data = Graph().parse(data="""
@prefix dbr: <http://dbpedia.org/resource/> .
@prefix dbo: <http://dbpedia.org/ontology/> .
@prefix ex: <http://example.org/> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .
dbr:The_Dark_Knight ex:releaseDateUSA "2008-07-18"^^xsd:date .
dbr:Interstellar dbo:releaseDate "2014-11-07"^^xsd:date .
""", format="turtle")
r3_schema_start = Graph()


### R3a — Inspect and predict

Complete this before adding a rule:

| Candidate | Already supplied? | My prediction after inference |
|---|---|---|
| The Dark Knight — releaseDateUSA — 2008-07-18 | … | … |
| The Dark Knight — releaseDate — 2008-07-18 | … | … |
| Interstellar — releaseDate — 2014-11-07 | … | … |
| Interstellar — releaseDateUSA — 2014-11-07 | … | … |

**Your reasoning:** Which direction would require extra knowledge about the country? …


<details>
<summary>Show prediction explanation — R3a</summary>

The supplied US-specific relationship should also support a general release-date relationship. The general relationship alone gives no evidence for the country-specific relationship. R3c checks both directions explicitly.

</details>


### R3b — Describe the property relationship

**Your code:** copy `r3_schema_start`, add the one subproperty declaration, and show that schema statement. This step changes the schema, not the supplied dates.


<details>
<summary>How to declare a subproperty</summary>

Use `RDFS.subPropertyOf` between property IRIs. A neutral pattern is `schema.add((EX.specialRelation, RDFS.subPropertyOf, EX.generalRelation))`.

The subject is the narrower property and the object the broader property. Inference carries existing subject–object pairs from the narrower relationship to the broader one. Copy the starting schema before adding your rule.

</details>


In [ ]:
# R3b — create r3_schema and display the subproperty declaration.
# Add your code here.


**Your observation — R3b**

Record the schema triple you added. Have you computed any new film-date statements at this step?

**My observation:** …


<details>
<summary>Show solution — R3b</summary>

```python
r3_schema=copy_graph(r3_schema_start)
r3_schema.add((EX.releaseDateUSA,RDFS.subPropertyOf,DBO.releaseDate))
display_rows([
    {"specific property":s.n3(r3_schema.namespace_manager),
     "relationship":p.n3(r3_schema.namespace_manager),
     "general property":o.n3(r3_schema.namespace_manager)}
    for s,p,o in r3_schema
],title="R3b: the supplied property relationship")
```

The row describes a relationship between predicates. It does not yet show an additional film-date statement; that requires the explicit inference step in R3c.

</details>


### R3c — Compute, compare and explain

**Requires:** R3b. Compute closure and inspect all four candidates from R3a. Preserve the typed date literals. If an SVG renderer is available, draw the date facts selected from the actual inferred graph.

**Your explanation:** Which statement is new? Why does the original US fact remain? How does this differ from replacing `releaseDate` with `releaseDateUSA` in the earlier RDF exercise? …


<details>
<summary>How to compare statements and show an inferred subgraph</summary>

The two graphs returned by `rdfs_closure` let you check the same candidate before and after expansion. Construct typed dates with `Literal("YYYY-MM-DD", datatype=XSD.date)`.

A small view should select triples that occur in `inferred`. Use a candidate list to choose what to inspect, and test membership before including any candidate in a view. `optional_svg(view, asserted_graph=asserted)` highlights additional edges when the earlier visualization setup is available; textual membership tables work on their own.

</details>


In [ ]:
# R3c — compute r3_asserted/r3_inferred, compare dates, and explain the direction.
# Add your code here.


**Your observation — R3c**

Identify the newly available date relationship, the preserved specific fact, and the reverse inference that is absent. Relate your table to the diagram when available.

**My interpretation and comparison with migration:** …


<details>
<summary>Show solution — R3c</summary>

```python
r3_asserted,r3_inferred=rdfs_closure(r3_data,r3_schema)
r3_us_date=Literal("2008-07-18",datatype=XSD.date)
r3_general_date=Literal("2014-11-07",datatype=XSD.date)
r3_candidates=[
    (DBR.The_Dark_Knight,EX.releaseDateUSA,r3_us_date),
    (DBR.The_Dark_Knight,DBO.releaseDate,r3_us_date),
    (DBR.Interstellar,DBO.releaseDate,r3_general_date),
    (DBR.Interstellar,EX.releaseDateUSA,r3_general_date),
]
show_memberships(r3_asserted,r3_inferred,r3_candidates,
                 title="R3c: specific and general release-date relationships")
r3_view=Graph()
for prefix,namespace in r3_inferred.namespaces():
    r3_view.bind(prefix,namespace)
for triple in r3_inferred:
    if triple[1] in {EX.releaseDateUSA,DBO.releaseDate}:
        r3_view.add(triple)
optional_svg(r3_view,asserted_graph=r3_asserted,
             title="Subproperty inference: actual computed date relationships")
assert r3_candidates[0] in r3_inferred
assert r3_candidates[1] not in r3_asserted and r3_candidates[1] in r3_inferred
assert r3_candidates[3] not in r3_inferred
```

The four after-inference statuses are True, True, True, False. Only the general date for The Dark Knight is newly derived among these candidates. The original subject and typed date are preserved; the relationship is generalized. In the diagram, the additional edge is orange/dashed. The graph used for that diagram contains only date facts actually found in the computed closure.

</details>


<details>
<summary>Original complete task description — R3</summary>

## Exercise R3 — A specific date is also a general date

**Goal:** Use a subproperty to obtain a more general view of the same data.

Exercise 8 in the RDF part changes releaseDate to releaseDateUSA. Such a migration is meaningful only when the dates really describe the selected US releases: renaming a property cannot establish that fact.

Here the preparation data explicitly supplies one selected US-release date and one date recorded only under the general releaseDate property.

**Tasks**

1. Declare ex:releaseDateUSA to be a subproperty of dbo:releaseDate.
2. Predict which additional date triple is entailed, then run inference.
3. Verify that the specific US-date assertion remains present.
4. Check whether Interstellar's general date now also appears under releaseDateUSA. Explain the direction of the rule.
5. Compare this approach with physically replacing predicates in the RDF graph.

**Hint:** Use RDFS.subPropertyOf. Inference adds general consequences; it does not erase the more specific input statement.

</details>


<details>
<summary>Complete reference solution — R3</summary>

The schema rule propagates from the more specific property to the more general property. The graph does not establish a country for the date recorded only under dbo:releaseDate.

```python
r3_schema = copy_graph(r3_schema_start)
r3_schema.add((EX.releaseDateUSA, RDFS.subPropertyOf, DBO.releaseDate))
r3_asserted, r3_inferred = rdfs_closure(r3_data, r3_schema)

r3_us_date = Literal("2008-07-18", datatype=XSD.date)
r3_general_date = Literal("2014-11-07", datatype=XSD.date)

assert (DBR.The_Dark_Knight, EX.releaseDateUSA, r3_us_date) in r3_inferred
assert (DBR.The_Dark_Knight, DBO.releaseDate, r3_us_date) not in r3_asserted
assert (DBR.The_Dark_Knight, DBO.releaseDate, r3_us_date) in r3_inferred
assert (DBR.Interstellar, EX.releaseDateUSA, r3_general_date) not in r3_inferred

r3_view = copy_graph(r3_data)
for triple in r3_inferred.triples((DBR.The_Dark_Knight, DBO.releaseDate, None)):
    r3_view.add(triple)
optional_svg(
    r3_view, asserted_graph=r3_asserted,
    title="Subproperty inference: US release date implies release date",
)
```

A consumer using the general property can see the inferred general assertion. The original US-specific fact is preserved. A data migration and an inference rule serve different purposes; the latter expresses a semantic relationship between the properties.

---

**How to read the code and its output**

`r3_asserted` contains the supplied dates and the subproperty declaration; `r3_inferred` is the expanded copy. `r3_us_date` and `r3_general_date` are the two typed literals used in the membership checks.

Read the four `assert` statements in order:

1. The Dark Knight's specific US-date assertion remains in `r3_inferred`.
2. Its general release-date assertion was **not** in `r3_asserted`.
3. That general assertion **is** in `r3_inferred`, so it was obtained by inference.
4. Interstellar's general date has not produced a US-specific assertion.

The relevant before-and-after state is:

| Statement | In the asserted union? | After inference? |
|---|---|---|
| The Dark Knight — releaseDateUSA — 2008-07-18 | Yes | Yes |
| The Dark Knight — releaseDate — 2008-07-18 | No | Yes |
| Interstellar — releaseDate — 2014-11-07 | Yes | Yes |
| Interstellar — releaseDateUSA — 2014-11-07 | No | No |

The Interstellar general-date row supplies context; the four checks above focus on the preserved specific fact, the newly derived general fact, and the missing reverse inference.

Successful `in` and `not in` assertions produce **no console output**. A false condition raises `AssertionError` and stops the cell. The expected visible result is the **SVG diagram** from `optional_svg(...)`. In `r3_view`, the two supplied date edges are grey; The Dark Knight's additional general-date edge is orange and dashed.

The subproperty rule preserves the subject, date value, and `xsd:date` datatype while adding the broader relationship. It does not infer a country from a general date. The original `r3_data` remains unchanged, while consumers of the expanded graph can use either the specific fact or its general consequence.

The optional SVG helper shows the diagram when its renderer is available; the membership checks work independently of visualization.

</details>


## Exercise R2 — What do relationships tell us about their participants?

**What is this about?** A relationship can imply a type for its subject or its object. We separate these effects, then watch a newly obtained type activate the class hierarchy you already know.

**After this exercise, you can:** distinguish domain from range by the end of a triple it affects; trace a multi-rule consequence; explain why typing a film does not supply a missing director.

**Given:** the relationships and existing superclass rules below. **Your result:** four short experiments with named schema versions and readable result tables. Start every expansion from `r2_data`, not from a previous expanded result.

**Reset:** rerun preparation, then R2a–d in order. Each schema copy records the next deliberate stage.


In [ ]:
r2_data = Graph().parse(data="""
@prefix dbr: <http://dbpedia.org/resource/> .
@prefix dbo: <http://dbpedia.org/ontology/> .
@prefix ex: <http://example.org/> .
dbr:The_Dark_Knight dbo:starring dbr:Heath_Ledger ;
                    dbo:director dbr:Christopher_Nolan .
ex:FilmWithoutDirector a dbo:Film .
""", format="turtle")

r2_schema_start = Graph().parse(data="""
@prefix dbo: <http://dbpedia.org/ontology/> .
@prefix ex: <http://example.org/> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
dbo:Actor rdfs:subClassOf dbo:Person .
dbo:Director rdfs:subClassOf dbo:Person .
dbo:Person rdfs:subClassOf ex:Agent .
""", format="turtle")


### R2a — Investigate the subject: domain only

Predict whether the subject of `starring` and its object gain types if we add **only** a Film domain for that property.

**Your code:** make `r2_domain_schema` from `r2_schema_start`, add that one domain statement, and compute `r2_domain_asserted/r2_domain_inferred`. Inspect The Dark Knight's Film membership and Heath Ledger's Actor membership.

**Your observation and rule:** …


<details>
<summary>How to use a domain declaration</summary>

A domain is a schema statement about a property: `(property_iri, RDFS.domain, class_iri)`. With an existing `(subject, property_iri, object)` fact, this rule classifies the **subject**.

Add the schema statement, then call `rdfs_closure(original_data, schema_version)`. This returns a fresh asserted union and expanded graph for this experiment. Reusing the data input avoids carrying conclusions over from an earlier schema.

</details>


In [ ]:
# R2a — add only the starring domain and inspect its effect.
# Add your code here.


**Your observation — R2a**

Which participant's selected type changed, and which did not? Identify the property use and schema declaration that support the change.

**My observation and rule:** …


<details>
<summary>Show solution — R2a</summary>

```python
r2_domain_schema=copy_graph(r2_schema_start)
r2_domain_schema.add((DBO.starring,RDFS.domain,DBO.Film))
r2_domain_asserted,r2_domain_inferred=rdfs_closure(r2_data,r2_domain_schema)
r2_domain_candidates=[
    (DBR.The_Dark_Knight,RDF.type,DBO.Film),
    (DBR.Heath_Ledger,RDF.type,DBO.Actor),
]
show_memberships(r2_domain_asserted,r2_domain_inferred,r2_domain_candidates,
                 title="R2a: a domain classifies the subject")
assert r2_domain_candidates[0] in r2_domain_inferred
assert r2_domain_candidates[1] not in r2_domain_inferred
```

The Film candidate changes to True; the Actor candidate remains False. The domain rule acts on the subject of the starring fact. There is not yet a premise classifying its object as an Actor.

</details>


### R2b — Investigate the object: add a range

**Predict first:** what happens to Heath Ledger when `starring` also has range Actor? Consider the supplied Actor → Person → Agent chain.

**Your code:** copy `r2_domain_schema` into `r2_range_schema`, add the range, and recompute from `r2_data`. Compare Ledger's three memberships after the domain-only stage and after this stage.

**Your explanation:** Name the premise responsible for each newly available type. …


<details>
<summary>How to use a range and compare experiment stages</summary>

A range statement has the form `(property_iri, RDFS.range, class_iri)`. It classifies the **object** of an existing property use.

For a stage comparison, test the same candidate in each named result graph. `display_rows` can show dictionaries with keys such as `"after domain"` and `"after range"`. Use these labels to keep a comparison between two inferred graphs distinct from an asserted/inferred comparison.

</details>


In [ ]:
# R2b — add the starring range and compare Ledger's types between stages.
# Add your code here.


**Your observation — R2b**

Compare all three Ledger memberships between the domain-only and domain-plus-range stages. Explain the first step and the subsequent chain.

**My observation and rule chain:** …


<details>
<summary>Show solution — R2b</summary>

```python
r2_range_schema=copy_graph(r2_domain_schema)
r2_range_schema.add((DBO.starring,RDFS.range,DBO.Actor))
r2_range_asserted,r2_range_inferred=rdfs_closure(r2_data,r2_range_schema)
r2_ledger_candidates=[
    (DBR.Heath_Ledger,RDF.type,DBO.Actor),
    (DBR.Heath_Ledger,RDF.type,DBO.Person),
    (DBR.Heath_Ledger,RDF.type,EX.Agent),
]
display_rows([
    {"statement":tuple(x.n3(r2_range_inferred.namespace_manager) for x in t),
     "after domain only":t in r2_domain_inferred,
     "after adding range":t in r2_range_inferred}
    for t in r2_ledger_candidates
],title="R2b: a range supplies the first type, then subclass rules apply")
assert all(t not in r2_domain_inferred and t in r2_range_inferred
           for t in r2_ledger_candidates)
```

The three rows change False → True. The range gives Ledger the Actor type directly. Actor-subclass-Person then yields Person, and Person-subclass-Agent yields Agent. The existing hierarchy becomes useful once the range establishes the first membership; the library follows all steps in the same expansion.

</details>


### R2c — Transfer the pattern to another relationship

Apply what you learned to `director`: its domain is Film and its range Director.

**Your code:** create `r2_complete_schema` from `r2_range_schema`, add both declarations, and compute a fresh closure. Show the newly established types of Christopher Nolan, then confirm all seven selected type statements in the complete exercise.

**Your explanation:** Which Film conclusion now has two supporting relationship uses? Does that create two stored copies? …


<details>
<summary>How to apply known rules to another property</summary>

Use the same domain/range pattern with another property IRI. Existing schema statements remain when you copy the previous schema version.

An RDF graph is a set: two reasoning paths can support the same triple without creating two copies. `all(candidate in graph for candidate in candidates)` checks a collection of predictions; display the individual statuses before using the assertion.

</details>


In [ ]:
# R2c — add the director declarations and inspect the complete selected result.
# Add your code here.


**Your observation — R2c**

Which additional types did Nolan acquire? Explain why the same Film conclusion can have two supporting paths without two stored copies.

**My interpretation:** …


<details>
<summary>Show solution — R2c</summary>

```python
r2_complete_schema=copy_graph(r2_range_schema)
r2_complete_schema.add((DBO.director,RDFS.domain,DBO.Film))
r2_complete_schema.add((DBO.director,RDFS.range,DBO.Director))
r2_complete_asserted,r2_complete_inferred=rdfs_closure(r2_data,r2_complete_schema)
r2_expected=[
    (DBR.The_Dark_Knight,RDF.type,DBO.Film),
    (DBR.Heath_Ledger,RDF.type,DBO.Actor),
    (DBR.Heath_Ledger,RDF.type,DBO.Person),
    (DBR.Heath_Ledger,RDF.type,EX.Agent),
    (DBR.Christopher_Nolan,RDF.type,DBO.Director),
    (DBR.Christopher_Nolan,RDF.type,DBO.Person),
    (DBR.Christopher_Nolan,RDF.type,EX.Agent),
]
show_memberships(r2_complete_asserted,r2_complete_inferred,r2_expected,
                 title="R2c: the seven selected complete-schema consequences")
r2_schema=copy_graph(r2_complete_schema)
r2_asserted,r2_inferred=r2_complete_asserted,r2_complete_inferred
assert all(t not in r2_asserted and t in r2_inferred for t in r2_expected)
```

All seven selected memberships are absent from the asserted data/schema union and present after inference. Nolan gains Director, Person and Agent. The two Film-domain rules both support The Dark Knight's one Film triple; the graph stores that triple once.

</details>


### R2d — Investigate what the schema leaves open

**Requires:** R2c. Retrieve all director values for `EX.FilmWithoutDirector`. Does the result contain a newly invented director? Is an application-validation report part of the helper's return values?

**Your code:** display the Film membership and the retrieved list. **Your explanation:** why does a domain declaration not make a relationship mandatory for every instance of its domain class? …


<details>
<summary>How to look up relationship values</summary>

RDFLib's `graph.objects(subject, predicate)` iterates the matching objects. Converting it with `list(...)` makes an empty result visible as `[]`.

`rdfs_closure` returns two graphs. It materializes consequences; it does not return a SHACL-style validation report or invent values to meet an application requirement.

</details>


In [ ]:
# R2d — inspect the FilmWithoutDirector relationship values.
# Add your code here.


**Your observation — R2d**

Record the retrieved director list. Explain what this result establishes and which application requirement it does not check.

**My observation and distinction:** …


<details>
<summary>Show solution — R2d</summary>

```python
r2_missing_values=list(r2_complete_inferred.objects(EX.FilmWithoutDirector,DBO.director))
display_rows([{
    "record":EX.FilmWithoutDirector.n3(r2_complete_inferred.namespace_manager),
    "is a Film":(EX.FilmWithoutDirector,RDF.type,DBO.Film) in r2_complete_inferred,
    "director values":r2_missing_values,
}],title="R2d: a type does not create a mandatory relationship")
assert r2_missing_values==[]
```

The table shows Film membership True and an empty director list. The rule describes subjects that already use a property; it does not demand a property use for every Film. A missing requirement is therefore not an RDFS validation finding. SHACL will express and test that application's expectation.

</details>


<details>
<summary>Original complete task description — R2</summary>

## Exercise R2 — Infer missing types from relationships

**Goal:** Combine domain, range and subclass rules.

The preparation cell has two film relationships with no explicit types for their participants. A separate film has a type but no director relationship.

**Tasks**

1. Copy r2_schema_start into r2_schema.
2. Give starring domain Film and range Actor.
3. Give director domain Film and range Director.
4. Predict and then verify the types of The Dark Knight, Heath Ledger and Christopher Nolan.
5. Inspect the film with no director. Has inference created a director relationship? Has it reported a missing-field error?

**Hints:** RDFS.domain and RDFS.range are properties of properties. Add their schema triples rather than adding the expected instance types manually.

</details>


<details>
<summary>Complete reference solution — R2</summary>

The property uses supply the first types; the existing class hierarchy supplies the more general types. No type assertions need to be inserted into the input data by hand.

```python
r2_schema = copy_graph(r2_schema_start)
r2_schema.add((DBO.starring, RDFS.domain, DBO.Film))
r2_schema.add((DBO.starring, RDFS.range, DBO.Actor))
r2_schema.add((DBO.director, RDFS.domain, DBO.Film))
r2_schema.add((DBO.director, RDFS.range, DBO.Director))
r2_asserted, r2_inferred = rdfs_closure(r2_data, r2_schema)

r2_expected = [
    (DBR.The_Dark_Knight, RDF.type, DBO.Film),
    (DBR.Heath_Ledger, RDF.type, DBO.Actor),
    (DBR.Heath_Ledger, RDF.type, DBO.Person),
    (DBR.Heath_Ledger, RDF.type, EX.Agent),
    (DBR.Christopher_Nolan, RDF.type, DBO.Director),
    (DBR.Christopher_Nolan, RDF.type, DBO.Person),
    (DBR.Christopher_Nolan, RDF.type, EX.Agent),
]
assert all(t not in r2_asserted and t in r2_inferred for t in r2_expected)
assert not list(r2_inferred.objects(EX.FilmWithoutDirector, DBO.director))

for triple in r2_expected:
    print(tuple(term.n3(r2_inferred.namespace_manager) for term in triple))
print("Director values for FilmWithoutDirector:",
      list(r2_inferred.objects(EX.FilmWithoutDirector, DBO.director)))
```

RDFS derives types from relationships that occur. It does not require every Film to have a director, invent a missing director, or return an application-validation report. We will express that kind of requirement in SHACL.

---

**How to read the code and its output**

`r2_expected` lists seven proposed consequences. The first assertion checks, for **every** listed triple, that it is absent from `r2_asserted` and present in `r2_inferred`. Thus it tests newly obtained information, not just the presence of facts that could already have been input.

The next loop prints each selected triple as a tuple of readable prefixed names. Read the three entries as **subject, predicate, object**. For example:

```text
('dbr:Heath_Ledger', 'rdf:type', 'dbo:Actor')
```

This means “Heath Ledger is an instance of Actor”; the third entry names the class. The seven printed tuples correspond to these reasoning paths:

| Resource | First type obtained | Further types obtained |
|---|---|---|
| The Dark Knight | Film, from the domain of either relationship | None among the selected Person/Agent types |
| Heath Ledger | Actor, from the range of starring | Person, then Agent |
| Christopher Nolan | Director, from the range of director | Person, then Agent |

There is one selected type for the film and three for each person. Both domains support the **same** Film triple, which occurs only once because RDF graphs are sets. The full closure also contains additional schema-related statements.

The final output is:

```text
Director values for FilmWithoutDirector: []
```

The empty list means no director value was found. The preceding `assert not list(...)` therefore succeeds silently. As with the first assertion, a failed condition would raise `AssertionError` and stop the cell; successful assertions do not print `True`.

A domain classifies subjects of existing property uses. It does not require every Film to use that property. Consequently, FilmWithoutDirector gains neither an invented director nor a missing-field report. That application requirement needs a separate SHACL constraint.

</details>


## Consolidation — combine the rules you have investigated

The complete original multi-rule worked example follows. Predict its selected results using the rules from R1, R3 and R2, then run the provided demonstration. Both visualizations remain available.


## Worked example — one relationship, several conclusions

This is a fully explained demonstration. First read the data and schema, and predict the types of **Batman Begins** and **Christian Bale**.

The supplied relationship has neither an explicit Film type for its subject nor an Actor type for its object. The schema describes the meaning of the property and the class hierarchy.


In [ ]:
walk_data = Graph().parse(data="""
@prefix dbr: <http://dbpedia.org/resource/> .
@prefix dbo: <http://dbpedia.org/ontology/> .
dbr:Batman_Begins dbo:starring dbr:Christian_Bale .
""", format="turtle")

walk_schema = Graph().parse(data="""
@prefix dbo: <http://dbpedia.org/ontology/> .
@prefix ex: <http://example.org/> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
dbo:starring rdfs:domain dbo:Film ; rdfs:range dbo:Actor .
dbo:Actor rdfs:subClassOf dbo:Person .
dbo:Person rdfs:subClassOf ex:Agent .
""", format="turtle")

walk_expected = [
    (DBR.Batman_Begins, RDF.type, DBO.Film),
    (DBR.Christian_Bale, RDF.type, DBO.Actor),
    (DBR.Christian_Bale, RDF.type, DBO.Person),
    (DBR.Christian_Bale, RDF.type, EX.Agent),
]

print("Before inference:", [triple in walk_data for triple in walk_expected])
walk_asserted, walk_inferred = rdfs_closure(walk_data, walk_schema)
print("After inference: ", [triple in walk_inferred for triple in walk_expected])
assert all(triple in walk_inferred for triple in walk_expected)
assert all(triple not in walk_asserted for triple in walk_expected)


The expected membership checks are **four False values before** and **four True values after** inference.

The reasoning chain is:

- The domain of starring gives the subject the Film type.
- Its range gives the object the Actor type.
- The subclass chain then gives the actor the Person and Agent types.

These are additional assertions that follow from the supplied schema. They were not downloaded from DBpedia.

The next cell shows only these selected film-domain triples. The full closure also contains general schema statements; displaying all of them would obscure this particular explanation.


In [ ]:
walk_new_triples = set(walk_inferred) - set(walk_asserted)
print("Selected new type statements:")
for triple in walk_expected:
    if triple in walk_inferred:
        print("  ", tuple(term.n3(walk_inferred.namespace_manager) for term in triple))

walk_view = copy_graph(walk_data)
for prefix, namespace in walk_asserted.namespaces():
    walk_view.bind(prefix, namespace)
for triple in walk_inferred:
    if triple in walk_expected:
        walk_view.add(triple)
optional_svg(
    walk_view,
    asserted_graph=walk_asserted,
    title="RDFS: original relationship and selected inferred types",
)


### Explore the same selected inferences with yFiles

The orange dashed edges are exactly the selected triples that were absent from the asserted data-and-schema union. Compare the interactive view with the SVG above; the full closure remains in `walk_inferred`.


In [ ]:
optional_yfiles(walk_view,asserted_graph=walk_asserted,layout="hierarchical")


### Reading the result and repeating an experiment

The result is a **materialized graph**: inferred statements are stored alongside the asserted statements. owlrl does not attach a proof or an asserted/inferred flag to every triple. We keep the asserted union separately and use a set difference to identify newly added triples.

After changing a premise or the schema, call rdfs_closure again on the original data and the updated schema. Deleting a premise from an already expanded graph does not automatically retract the conclusions that were added earlier.

For an exercise, checking selected triples is more useful than expecting a fixed total closure size.

RDFS reasoning also follows an open-world reading: an absent statement is not automatically false. Domain and range describe consequences of existing property uses; they do not require such a property to occur.


**Check the display against computation.** The selected edges above are included only if they occur in `walk_inferred`; the candidate list is a filter, not a source of invented results. If you change a premise, recompute from the asserted data and inspect which memberships actually remain.

**Short interpretation:** Is a surprising inferred type a Turtle syntax error, a failed Python assertion, or a consequence of the supplied schema? Explain the difference before continuing. …


## Exercise R4 — Which schema premise explains an unintended conclusion?

**What is this about?** A reasoner can apply your rules correctly and still derive classifications you did not intend. You will diagnose the responsible premise, change the schema, and recompute its consequences.

**After this exercise, you can:** explain global domain and conjunctive ranges using concrete facts; repair a schema without editing the data; distinguish unintended inference from missing application requirements.

Complete **A1 → A2** before **B1 → B2**. The cases have independent preparation. **Your result:** two before/after comparisons and one explanation of the limits of RDFS.

**Reset:** rerun the relevant case preparation, then its diagnosis and repair. Always recompute from that case's original data.


### Case A — Provided preparation

One name property describes both a film and an actor. Read its domain and predict the effect on each subject.


In [ ]:
r4_global_data = Graph().parse(data="""
@prefix dbr: <http://dbpedia.org/resource/> .
@prefix dbo: <http://dbpedia.org/ontology/> .
@prefix ex: <http://example.org/> .
dbr:Batman_Begins a dbo:Film ; ex:name "Batman Begins" .
dbr:Christian_Bale a dbo:Actor ; ex:name "Christian Bale" .
""", format="turtle")
r4_global_schema = Graph().parse(data="""
@prefix dbo: <http://dbpedia.org/ontology/> .
@prefix ex: <http://example.org/> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
dbo:Actor rdfs:subClassOf dbo:Person .
ex:name rdfs:domain dbo:Person .
""", format="turtle")


### R4a1 — Observe and locate the responsible premise

**Your code:** compute `r4_a_asserted/r4_a_inferred` and display Person membership for Batman Begins and Bale.

**Your diagnosis:** identify the exact data triple and schema triple that together establish the surprising classification. …


<details>
<summary>How to trace an unexpected type</summary>

Start with the candidate conclusion and ask which property use and domain or range statement could support it. Display the candidate before/after inference. The reasoner does not know that two class names should denote incompatible kinds of thing merely from their names.

</details>


In [ ]:
# R4a1 — infer case A and identify its supporting premises.
# Add your code here.


**Your observation — R4a1**

Name the surprising conclusion, its supporting data triple, and the responsible schema premise.

**My diagnosis:** …


<details>
<summary>Show solution — R4a1</summary>

```python
r4_a_asserted,r4_a_inferred=rdfs_closure(r4_global_data,r4_global_schema)
r4_a_candidates=[
    (DBR.Batman_Begins,RDF.type,DBO.Person),
    (DBR.Christian_Bale,RDF.type,DBO.Person),
]
show_memberships(r4_a_asserted,r4_a_inferred,r4_a_candidates,
                 title="R4a1: global domain consequences")
assert all(t in r4_a_inferred for t in r4_a_candidates)
```

Both Person candidates are derived. Batman Begins uses EX.name, and EX.name has Person as its domain. The domain applies to every subject using that property. This is a correctly applied schema rule, not a parser error.

</details>


### R4a2 — Repair the schema and verify the effect

**Requires:** R4a1. Remove the inappropriate Person domain from a schema copy and recompute from `r4_global_data`. Keep all input data.

**Your code:** compare the two Person memberships before and after the schema repair. **Your explanation:** why can Bale remain a Person even after that domain is removed? …


<details>
<summary>How to remove a schema statement and recompute</summary>

`graph.remove((subject, predicate, object))` removes matching triples. Use a copied schema so that the original remains available for comparison. Re-run `rdfs_closure(original_data, corrected_schema)`.

Removing a premise from an already materialized graph does not automatically retract the old consequences. A new closure from the original assertions tests the repaired schema.

</details>


In [ ]:
# R4a2 — remove the domain in a copy and recompute case A.
# Add your code here.


**Your observation — R4a2**

Compare the film and actor memberships before and after repair. Which remaining premise accounts for the actor's classification?

**My observation and justification:** …


<details>
<summary>Show solution — R4a2</summary>

```python
r4_a_fixed_schema=copy_graph(r4_global_schema)
r4_a_fixed_schema.remove((EX.name,RDFS.domain,DBO.Person))
_,r4_a_fixed=rdfs_closure(r4_global_data,r4_a_fixed_schema)
display_rows([
    {"statement":tuple(x.n3(r4_a_inferred.namespace_manager) for x in t),
     "original schema":t in r4_a_inferred,"repaired schema":t in r4_a_fixed}
    for t in r4_a_candidates
],title="R4a2: compare independent closures after the repair")
assert r4_a_candidates[0] not in r4_a_fixed
assert r4_a_candidates[1] in r4_a_fixed
```

The film's Person membership changes True → False because its supporting domain premise has gone. Bale's remains True because the independent Actor-subclass-Person premise still applies to Bale's asserted Actor type. The False status is absence of derivation in the corrected closure, not an explicit negative RDF statement.

</details>


### Case B — Provided preparation

Start this independent case now. The modeller intended an involved person to be an Actor **or** a Director. Predict what the two range declarations actually mean.


In [ ]:
r4_range_data = Graph().parse(data="""
@prefix dbr: <http://dbpedia.org/resource/> .
@prefix ex: <http://example.org/> .
dbr:Batman_Begins ex:involvedPerson dbr:Christian_Bale .
""", format="turtle")
r4_range_schema = Graph().parse(data="""
@prefix dbo: <http://dbpedia.org/ontology/> .
@prefix ex: <http://example.org/> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
dbo:Actor rdfs:subClassOf dbo:Person .
dbo:Director rdfs:subClassOf dbo:Person .
ex:involvedPerson rdfs:range dbo:Actor, dbo:Director .
""", format="turtle")


### R4b1 — Diagnose the two-range result

**Your code:** compute `r4_b_asserted/r4_b_inferred` and display Bale's Actor, Director and Person memberships.

**Your explanation:** give one inference justification for each of the two specific role types. Why are the ranges not alternatives? …


<details>
<summary>How to read multiple domain or range declarations</summary>

A comma-separated Turtle object list represents several triples. Each applicable range triple can independently supply an object type. Collecting conclusions does not choose one of those rules as an alternative.

Use the same candidate-membership table as in case A and read every row before changing the schema.

</details>


In [ ]:
# R4b1 — infer case B and display the three selected memberships.
# Add your code here.


**Your observation — R4b1**

Record the three selected memberships. Explain how each specific role follows, and why these premises do not offer an either/or choice.

**My diagnosis:** …


<details>
<summary>Show solution — R4b1</summary>

```python
r4_b_asserted,r4_b_inferred=rdfs_closure(r4_range_data,r4_range_schema)
r4_b_candidates=[
    (DBR.Christian_Bale,RDF.type,DBO.Actor),
    (DBR.Christian_Bale,RDF.type,DBO.Director),
    (DBR.Christian_Bale,RDF.type,DBO.Person),
]
show_memberships(r4_b_asserted,r4_b_inferred,r4_b_candidates,
                 title="R4b1: two ranges supply two object types")
assert all(t in r4_b_inferred for t in r4_b_candidates)
```

All three rows change False → True. The Actor and Director range triples both apply to the same involvedPerson value. Either role's superclass rule also supports Person. RDFS does not interpret the two ranges as an either/or choice.

</details>


### R4b2 — Generalize the range and compare

**Requires:** R4b1. Replace both specific ranges with Person in a schema copy and recompute from `r4_range_data`.

**Your code:** compare the three memberships before and after repair. **Your explanation:** does this replacement establish that every involved person is exactly an Actor or a Director? Why is a fresh closure necessary? …

**Final reflection:** Would either repaired RDFS schema reject a Film that has no name or director? What additional kind of requirement would we need? …


<details>
<summary>How to remove every value for a schema predicate</summary>

`schema.remove((property_iri, RDFS.range, None))` removes all range statements for the chosen property; `None` is a wildcard. Add the intended replacement range afterwards.

Keep the original and repaired expanded graphs under different names. Compare the same candidates in both; use original data for both closure computations.

</details>


In [ ]:
# R4b2 — repair case B, compare memberships, and explain RDFS's limits.
# Add your code here.


**Your observation — R4b2**

Compare the selected types across the two closures. Explain what the weaker range leaves open and why it still does not impose a required field.

**My observation and conclusion:** …


<details>
<summary>Show solution — R4b2</summary>

```python
r4_b_fixed_schema=copy_graph(r4_range_schema)
r4_b_fixed_schema.remove((EX.involvedPerson,RDFS.range,None))
r4_b_fixed_schema.add((EX.involvedPerson,RDFS.range,DBO.Person))
_,r4_b_fixed=rdfs_closure(r4_range_data,r4_b_fixed_schema)
display_rows([
    {"statement":tuple(x.n3(r4_b_inferred.namespace_manager) for x in t),
     "original schema":t in r4_b_inferred,"repaired schema":t in r4_b_fixed}
    for t in r4_b_candidates
],title="R4b2: the repaired range leaves specific roles open")
assert r4_b_candidates[0] not in r4_b_fixed
assert r4_b_candidates[1] not in r4_b_fixed
assert r4_b_candidates[2] in r4_b_fixed
```

Actor and Director change True → False; Person stays True. The weaker range establishes the shared type and leaves the specific role open. It does not express a union of exactly those two roles. Recomputing avoids retaining the previously materialized role triples. Neither schema states a required name/director count; a SHACL shape will express that application requirement.

</details>


<details>
<summary>Original complete task description — R4</summary>

## Exercise R4 — Find and repair a modelling mistake

**Goal:** Diagnose surprising consequences without treating RDFS as a validator.

The two preparation cases below are deliberately independent.

**Case A — A global domain**

A single name property is used for a film and an actor. The schema assigns that property domain Person.

**Case B — Two ranges**

The property involvedPerson has both Actor and Director as ranges. The modeller intended “actor or director”.

**Tasks**

1. Predict the new type of Batman Begins in A and the new types of Christian Bale in B.
2. Run inference for each case and confirm the consequences.
3. Explain why both range declarations apply together.
4. Repair A so that naming a film does not type it as a Person.
5. Repair B using the shared superclass Person. Recompute on fresh graph copies and verify that the earlier Actor/Director conclusions are no longer present in B.
6. Would RDFS reject a Film that has no name or director? Explain why this motivates application-specific SHACL constraints.

**Hint:** Remove the inappropriate schema premises and recompute from the asserted data. Removing them from an already expanded graph does not undo previously stored consequences.

</details>


<details>
<summary>Complete reference solution — R4</summary>

In A, every subject using name acquires the Person type, including the film. In B, each object acquires both range types. RDFS properties are global; the two ranges mean “and”. Class names alone do not make Film and Person, or Actor and Director, disjoint.

```python
r4_a_asserted, r4_a_inferred = rdfs_closure(r4_global_data, r4_global_schema)
r4_b_asserted, r4_b_inferred = rdfs_closure(r4_range_data, r4_range_schema)

assert (DBR.Batman_Begins, RDF.type, DBO.Person) in r4_a_inferred
assert (DBR.Christian_Bale, RDF.type, DBO.Actor) in r4_b_inferred
assert (DBR.Christian_Bale, RDF.type, DBO.Director) in r4_b_inferred

# A: allow this broadly used name property without the Person domain.
r4_a_fixed_schema = copy_graph(r4_global_schema)
r4_a_fixed_schema.remove((EX.name, RDFS.domain, DBO.Person))
_, r4_a_fixed = rdfs_closure(r4_global_data, r4_a_fixed_schema)
assert (DBR.Batman_Begins, RDF.type, DBO.Person) not in r4_a_fixed
assert (DBR.Christian_Bale, RDF.type, DBO.Person) in r4_a_fixed

# B: infer only the shared superclass, not both specific roles.
r4_b_fixed_schema = copy_graph(r4_range_schema)
r4_b_fixed_schema.remove((EX.involvedPerson, RDFS.range, None))
r4_b_fixed_schema.add((EX.involvedPerson, RDFS.range, DBO.Person))
_, r4_b_fixed = rdfs_closure(r4_range_data, r4_b_fixed_schema)
assert (DBR.Christian_Bale, RDF.type, DBO.Person) in r4_b_fixed
assert (DBR.Christian_Bale, RDF.type, DBO.Actor) not in r4_b_fixed
assert (DBR.Christian_Bale, RDF.type, DBO.Director) not in r4_b_fixed
print("Both repaired schemas give the intended selected consequences.")
```

The repaired range leaves open which specific role the person has. It does not express a union of exactly Actor and Director. Missing names or directors still do not cause RDFS validation errors. SHACL lets us state and check those application requirements explicitly.

**Discussion:** Which statements were asserted, which were inferred, and which expectations exist only in the application requirements?

---

**How to read the code and its output**

The first two calls compute the original consequences in `r4_a_inferred` and `r4_b_inferred`. The following three assertions check the surprising results: Batman Begins becomes a Person in A, and Bale becomes both Actor and Director in B.

Next, the solution edits **schema copies**. For A it removes the name property's Person domain, then recomputes `r4_a_fixed`. For B it replaces both specific ranges with Person and recomputes `r4_b_fixed`. The membership tests compare the intended effects:

| Case | Result with the original schema | Result after repair and fresh inference |
|---|---|---|
| A: global name domain | Batman Begins is both Film and Person | Film remains; Person is no longer derived for the film |
| B: two ranges | Bale is Actor and Director, and therefore Person | Person is derived; neither specific role is derived |

In A, every subject using the name property is affected by its domain. No rule makes Film and Person disjoint. Bale remains a Person after repair through the independent Actor-subclass-Person rule.

In B, the comma-separated ranges are two applicable schema triples, so their classifications accumulate. Replacing both with Person weakens the conclusion and leaves the specific role undetermined.

Each `assert` is silent when its condition is `True`. A failed `in` or `not in` test raises `AssertionError` and stops the cell before its final message:

```text
Both repaired schemas give the intended selected consequences.
```

That message confirms the selected checks passed; it does not certify all facts or required fields. Recomputing is essential because old materialized type triples would remain in an already expanded graph after deleting their former premises. Fresh closure tests the corrected schema rather than retaining those stored conclusions.

</details>


### RDFS recap and transition to SHACL

A schema can generate additional type and relationship statements. Inference answers **“What follows from these premises?”**

Our next question is different: **“Does this supplied graph meet the requirements of our application?”** For example, our film application might require a selected US-release date even though RDF and RDFS allow us to leave it unspecified.

Keep the RDFS exercise graphs separate from the SHACL examples. Each part provides its own fixed starting data, so the results do not depend on which earlier exercise cells you have completed.


<a id="shacl-entry"></a>

# Part III — SHACL: checking application requirements

RDFS helped us derive additional statements. We now ask whether a supplied graph meets the requirements of a particular application.

**SHACL** (Shapes Constraint Language) expresses those requirements as RDF. We keep three roles separate:

| Graph | Role |
|---|---|
| Data graph | The film records being checked. |
| Shapes graph | Targets and constraints, such as a required date or an allowed value range. |
| Schema graph | Optional RDFS knowledge supplied explicitly when we study inference. |

The exercises use small, independent film datasets. Their outcomes do not depend on which facts you added, migrated, or deleted in the earlier RDF exercises. Our requirements are teaching examples, not universal rules about all film databases.

Workflow: **predict → validate → explain the report → repair a copy → validate again**. A validator reports problems; it does not automatically choose a correct date or repair the dataset.

References: [W3C SHACL](https://www.w3.org/TR/shacl/) and [pySHACL Python usage](https://github.com/RDFLib/pySHACL#python-module-use).


### Start this section — provided preparation

**Run the next cell as supplied.** You can start here in a fresh kernel without executing any earlier exercise solutions. It imports RDF terms, defines the prefixes and supplies graph-copying and table helpers. The following section-specific helper cell supplies inference or validation functions.

The setup code is infrastructure and may appear collapsed in Jupyter. Its usage is explained in the local **How to…** folds. Complete only cells labelled **Your code / Student work**; use the separate answer fields for predictions and observations.

**Optional views:** SVG and yFiles examples are retained. To enable them, run the helper-definition cells under **Additional RDF views** earlier in this notebook. Core tables, inference and validation work without those renderers. Reset an experiment by rerunning its supplied preparation, then the required subparts in order.


In [ ]:
# PROVIDED PREPARATION — run as supplied. Safe to rerun.
# No earlier exercise solution or visualization package is required.
import importlib.util
import shutil
from html import escape as html_escape
from rdflib import Graph, Namespace, URIRef, BNode, Literal
from rdflib.namespace import RDF, RDFS, XSD
from IPython.display import HTML, display

DBR = Namespace("http://dbpedia.org/resource/")
DBO = Namespace("http://dbpedia.org/ontology/")
EX = Namespace("http://example.org/")

def copy_graph(source):
    """Copy triples and prefix bindings; leave the original graph unchanged."""
    result = Graph()
    for prefix, namespace in source.namespaces():
        result.bind(prefix, namespace)
    for triple in source:
        result.add(triple)
    return result

def display_rows(rows, columns=None, title=""):
    """Display supplied result records as a readable, escaped HTML table."""
    rows = list(rows)
    if title:
        display(HTML("<p><strong>" + html_escape(str(title)) + "</strong></p>"))
    if not rows:
        print("No rows.")
        return
    columns = list(columns) if columns is not None else list(rows[0])
    header = "<tr>" + "".join("<th>" + html_escape(str(c)) + "</th>" for c in columns) + "</tr>"
    body = []
    for row in rows:
        body.append("<tr>" + "".join("<td>" + html_escape(str(row.get(c, "—"))) + "</td>" for c in columns) + "</tr>")
    display(HTML("<table><thead>" + header + "</thead><tbody>" + "".join(body) + "</tbody></table>"))

def show_memberships(asserted_graph, inferred_graph, candidates, title="Selected type statements"):
    """Show actual graph membership before/after inference, not expected answers."""
    rows = []
    for triple in candidates:
        rows.append({
            "Statement (subject, predicate, object)": " ".join(t.n3(inferred_graph.namespace_manager) for t in triple),
            "Asserted": triple in asserted_graph,
            "After inference": triple in inferred_graph,
        })
    display_rows(rows, title=title)

def optional_svg(graph, asserted_graph=None, focus_nodes=None, title="RDF graph"):
    """Display an available SVG helper; optional views never supply evidence."""
    renderer = globals().get("show_rdf_graphviz")
    if renderer is None:
        print("Optional SVG view: run the earlier Additional RDF views helper-definition cells to enable it. The tables above contain the computation results.")
        return
    if shutil.which("dot") is None:
        print("Optional SVG view needs Graphviz dot. The tables above remain available.")
        return
    renderer(graph, asserted_graph=asserted_graph, focus_nodes=focus_nodes, title=title)

def optional_yfiles(graph, asserted_graph=None, focus_nodes=None, layout="hierarchical"):
    """Display an available yFiles adapter, preserving the textual learning path."""
    adapter = globals().get("rdf_to_yfiles")
    if adapter is None or importlib.util.find_spec("yfiles_jupyter_graphs") is None:
        print("Optional yFiles view: install its package and run the earlier Additional RDF views helper-definition cells to enable it.")
        return
    widget = adapter(graph, asserted_graph=asserted_graph, focus_nodes=focus_nodes, layout=layout)
    display(widget)

print("Core preparation ready: RDF terms, namespaces, graph copying and result tables.")
print("Supplied helpers: copy_graph, display_rows, show_memberships, optional_svg, optional_yfiles.")


### Start this section

Run the supplied common section preparation and the SHACL helper cell below. The new section can start in a fresh kernel without completing earlier student exercises. `validate()` is a pySHACL library function; `parse_shacl_turtle()`, `print_shacl_results()`, `shacl_components()` and the comparison function are supplied notebook helpers.

**Progression:** S1 → S2 → S3 → basic S5 → S4 → qualified-language extension. Original identifiers stay unchanged. Targets and inference form the integration exercise after ordinary value constraints.

**Cell labels:** provided preparation = run as supplied; student work = write code; prediction/observation = explain in words; How to = operational help; Show solution = reveal after attempting the step. All original descriptions and complete reference solutions remain available.


In [ ]:
from pyshacl import validate

SH = Namespace("http://www.w3.org/ns/shacl#")

# Shared only within this SHACL section: parse self-contained Turtle fragments.
SHACL_PREFIXES = """
@prefix ex:   <http://example.org/> .
@prefix dbr:  <http://dbpedia.org/resource/> .
@prefix dbo:  <http://dbpedia.org/ontology/> .
@prefix rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix xsd:  <http://www.w3.org/2001/XMLSchema#> .
@prefix sh:   <http://www.w3.org/ns/shacl#> .
"""

def parse_shacl_turtle(body):
    """Create a new graph; every call starts with independent data."""
    return Graph().parse(data=SHACL_PREFIXES + body, format="turtle")

def shacl_result_rows(report_graph):
    """Read the report as RDF, without requiring SPARQL or pandas."""
    def readable(term):
        if term is None:
            return "—"
        return term.n3(report_graph.namespace_manager)

    rows = []
    for result in report_graph.subjects(RDF.type, SH.ValidationResult):
        rows.append({
            "focus node": readable(report_graph.value(result, SH.focusNode)),
            "path": readable(report_graph.value(result, SH.resultPath)),
            "value": readable(report_graph.value(result, SH.value)),
            "constraint": readable(report_graph.value(result, SH.sourceConstraintComponent)),
            "message": str(report_graph.value(result, SH.resultMessage) or ""),
        })
    return sorted(rows, key=lambda row: tuple(row.values()))

def print_shacl_results(report_graph):
    rows = shacl_result_rows(report_graph)
    if not rows:
        print("No validation results.")
    for row in rows:
        print(" | ".join(f"{key}: {value}" for key, value in row.items()))

def shacl_components(report_graph):
    return set(report_graph.objects(None, SH.sourceConstraintComponent))


In [ ]:
def compare_shacl_cases(cases, shapes, *, inference="none", schema=None):
    """Provided scaffold: validate copies and preserve every report."""
    results = {}
    for name, data in cases.items():
        conforms, report, text = validate(
            copy_graph(data), shacl_graph=shapes,
            ont_graph=copy_graph(schema) if schema is not None else None,
            inference=inference,
        )
        results[name] = {"conforms": bool(conforms), "report": report,
                         "text": text, "components": shacl_components(report)}
    display_rows([
        {"case": name, "conforms": r["conforms"],
         "findings": len(shacl_result_rows(r["report"]))}
        for name, r in results.items()
    ], title="Validation comparison")
    for name, result in results.items():
        print("\nCASE:", name, "| conforms:", result["conforms"])
        print_shacl_results(result["report"])
    return results


<details>
<summary>How to — read results and distinguish execution problems</summary>

`validate(data, shacl_graph=shapes, inference="none")` returns `(conforms, report_graph, report_text)`. The provided comparison helper calls it on fresh data copies and stores results by case name: `results["case"]["conforms"]` and `results["case"]["report"]` remain inspectable after another call.

Read **focus node → path → value → constraint → message** together. A count-related finding may have no individual value. `assert condition` is silent on success; an unsuccessful assertion raises `AssertionError` and stops its cell.

A Python or Turtle parse error prevents processing as written. A SHACL finding means the RDF was processed but did not satisfy a selected requirement. A failed assertion means a prediction/check did not match the result: inspect the visible evidence before changing anything.

Conformance concerns selected records and expressed constraints. It does not establish the historical accuracy of film facts. Label partial shapes as partial profiles: a value can satisfy the checks already added and fail a later requirement.

</details>


## Worked example — call the validator and read its result

This first example is complete so that you can learn the API before writing a shape.

The application requires at least one label for every targeted film. `sh:targetClass` selects the film records, `sh:path` selects their labels, and `sh:minCount 1` requires a value.

`validate(...)` returns three values:

1. `conforms`: the overall Boolean outcome.
2. `report_graph`: an RDF graph describing validation results.
3. `report_text`: a human-readable rendering of that report.

We explicitly start with `inference="none"`. Additional RDFS inference will be a separate experiment in S4.


In [ ]:
demo_data = parse_shacl_turtle("""
dbr:Batman_Begins a dbo:Film ;
    rdfs:label "Batman Begins"@en .
""")

demo_shapes = parse_shacl_turtle("""
ex:LabelDemoShape a sh:NodeShape ;
    sh:targetClass dbo:Film ;
    sh:property [
        sh:path rdfs:label ;
        sh:minCount 1 ;
        sh:message "A film record needs a label."
    ] .
""")

demo_conforms, demo_report, demo_text = validate(
    copy_graph(demo_data),
    shacl_graph=demo_shapes,
    inference="none",
)
print("Complete record conforms:", demo_conforms)
assert demo_conforms

# Remove a value from an independent copy, then validate that copy.
demo_missing_label = copy_graph(demo_data)
demo_missing_label.remove((DBR.Batman_Begins, RDFS.label, None))

demo_conforms, demo_report, demo_text = validate(
    demo_missing_label,
    shacl_graph=demo_shapes,
    inference="none",
)
print("Record without a label conforms:", demo_conforms)
print(demo_text)
print_shacl_results(demo_report)
assert not demo_conforms
assert SH.MinCountConstraintComponent in shacl_components(demo_report)


The missing-label report identifies `dbr:Batman_Begins` as its **focus node**, `rdfs:label` as the **path**, and the minimum-count constraint as the failed rule. A missing value does not have a `sh:value`; the reporting helper displays a dash.

The RDF syntax is still valid. The supplied record simply fails this application's requirement. Repairing the record means adding a suitable fact, then running validation again.

The following SVG highlights the affected focus node. The report remains the explanation of which constraint failed.


In [ ]:
optional_svg(
    demo_missing_label,
    focus_nodes={DBR.Batman_Begins},
    title="SHACL worked example: film record without a label",
)


## S1 — When is a date complete and correctly represented?

**What is this about?** A valid RDF record can fail an application's date requirements. Separate presence, number of values and representation before combining them.

**After this exercise, you can:**

- Translate the three date requirements into constraints.
- Locate a failed requirement in its report.
- Repair copies and explain the result.

**Given:** Four independent graphs in `s1_cases`.

**Your result:** Partial/complete shapes, a prediction/observation table and verified repairs.

**Application requirements:** For this application's selected film records, require exactly one `ex:releaseDateUSA` value with datatype `xsd:date`. Other countries' dates may exist separately. S1a and S1b deliberately implement only part of this final profile.

Work through the lettered parts in order. Rerun the supplied preparation cell to reset this exercise. Earlier exercises' student answers are not required.


<details>
<summary>Original complete task description — S1</summary>

## Exercise S1 — write a FilmShape for US release dates

**Goal:** build and use a property shape, and distinguish three kinds of violation.

For **this application**, each targeted film record must have exactly one selected US release date, represented by `ex:releaseDateUSA` and an `xsd:date` literal. This is a local application rule; films may have other release dates in other countries.

**Tasks**

1. Write `s1_shapes` as a separate RDF graph containing a `sh:NodeShape` targeted at `dbo:Film`.
2. Give its property shape the path `ex:releaseDateUSA`, minimum and maximum counts of 1, and datatype `xsd:date`.
3. Before executing, predict the result for each of the four datasets below.
4. Validate each dataset with `inference="none"`. Read the focus node, path, and failed constraint.
5. Repair a fresh copy of each invalid record and confirm that it now conforms.

**Hints:** `sh:property [ ... ]` creates a blank-node property shape; use `sh:path`, `sh:minCount`, `sh:maxCount`, and `sh:datatype`. A plain string such as `"2008-07-18"` has datatype `xsd:string`, even if its characters resemble a date.

</details>


In [ ]:
# Each case is a separate data graph containing one explicitly typed film.
s1_cases = {
    "valid": parse_shacl_turtle("""
        dbr:The_Dark_Knight a dbo:Film ;
            ex:releaseDateUSA "2008-07-18"^^xsd:date .
    """),
    "missing": parse_shacl_turtle("""
        dbr:The_Dark_Knight a dbo:Film .
    """),
    "string": parse_shacl_turtle("""
        dbr:The_Dark_Knight a dbo:Film ;
            ex:releaseDateUSA "2008-07-18" .
    """),
    "two_dates": parse_shacl_turtle("""
        dbr:The_Dark_Knight a dbo:Film ;
            ex:releaseDateUSA "2008-07-18"^^xsd:date,
                             "2008-07-22"^^xsd:date .
    """),
}


### S1a — Require a value

**Question:** What does presence alone establish?

**Use:** `s1_cases`; no prior student code.

**Task:** Create the partial `s1_minimum_shapes` with a film target, date path and minimum count. Validate the two cases.


**Predict before running:** Compare the valid and missing records. Count their date values.

**My prediction and reason:** _Write here._


<details>
<summary>How to — Require a value</summary>

`parse_shacl_turtle(body)` adds the supplied prefixes and returns a new RDFLib graph. A node shape has `sh:targetClass`; a property shape has `sh:path` and constraints. Construct a complete Turtle fragment for each stage so anonymous property shapes do not accumulate accidentally. The comparison helper supplies repetitive execution; you supply the requirements.

**Run one case with the supplied comparison helper**

```python
results = compare_shacl_cases({"chosen_case": data_graph}, shapes_graph)
print(results["chosen_case"]["conforms"])
print_shacl_results(results["chosen_case"]["report"])
```

Replace the generic `data_graph` and `shapes_graph` with the graph and shape variables for your current experiment. A dictionary can contain one case or several; its keys label the displayed cases. The helper already displays the comparison table and report fields, so the last two lines are optional examples of inspecting a saved result later.

The return value is a dictionary keyed by case name. Each value is another dictionary containing `conforms` (Boolean), `report` (RDF graph), `text` (readable report), and `components` (set of failed constraint-component IRIs). For an error, read focus node, path, value and component together; a count finding can lack an individual value.

</details>


In [ ]:
# S1a — complete your code here.
# Keep earlier results under their own names.


**Observe and explain:** Record the actual result, identify the relevant rule or constraint, and explain your comparison with the prediction.

**My observation and explanation:** _Write here._


<details>
<summary>Show solution — S1a</summary>

```python
s1_minimum_shapes = parse_shacl_turtle("""
ex:DatePresenceShape a sh:NodeShape ;
    sh:targetClass dbo:Film ;
    sh:property [ sh:path ex:releaseDateUSA ; sh:minCount 1 ] .
""")
s1_presence_results = compare_shacl_cases({"valid": s1_cases["valid"], "missing": s1_cases["missing"]}, s1_minimum_shapes)

assert s1_presence_results["valid"]["conforms"]
assert not s1_presence_results["missing"]["conforms"]
```

The table shows True for valid and False for missing. The missing case identifies the film, date path and minimum-count component; there is no offending date value because its count is zero. This **partial profile** does not yet restrict datatype or maximum count. Assertions silently check the displayed evidence.

</details>


### S1b — Require exactly one value

**Question:** Can correctly typed dates fail a count requirement?

**Use:** `s1_cases`; build a fresh shape after S1a.

**Task:** Create `s1_cardinality_shapes` with both count conditions, but no datatype condition yet. Compare valid and two_dates.


**Predict before running:** What changes when the two-date record is checked against a maximum count?

**My prediction and reason:** _Write here._


<details>
<summary>How to — Require exactly one value</summary>

`parse_shacl_turtle(body)` adds the supplied prefixes and returns a new RDFLib graph. A node shape has `sh:targetClass`; a property shape has `sh:path` and constraints. Construct a complete Turtle fragment for each stage so anonymous property shapes do not accumulate accidentally. The comparison helper supplies repetitive execution; you supply the requirements.

Combining `sh:minCount 1` and `sh:maxCount 1` requires exactly one distinct value.

</details>


In [ ]:
# S1b — complete your code here.
# Keep earlier results under their own names.


**Observe and explain:** Record the actual result, identify the relevant rule or constraint, and explain your comparison with the prediction.

**My observation and explanation:** _Write here._


<details>
<summary>Show solution — S1b</summary>

```python
s1_cardinality_shapes = parse_shacl_turtle("""
ex:DateCardinalityShape a sh:NodeShape ;
    sh:targetClass dbo:Film ;
    sh:property [ sh:path ex:releaseDateUSA ; sh:minCount 1 ; sh:maxCount 1 ] .
""")
s1_cardinality_results = compare_shacl_cases({"valid": s1_cases["valid"], "two_dates": s1_cases["two_dates"]}, s1_cardinality_shapes)

assert s1_cardinality_results["valid"]["conforms"]
assert SH.MaxCountConstraintComponent in s1_cardinality_results["two_dates"]["components"]
```

The two-date record has two different values and fails maximum count, despite both being dates. Its representation is not the reason for this finding. The stage is still partial because the datatype requirement has not been added.

</details>


### S1c — Check the datatype

**Question:** Does date-looking text count as an RDF date?

**Use:** `s1_cases`; build the complete profile.

**Task:** Create complete `s1_shapes` with minimum count, maximum count and datatype. Compare valid and string.


**Predict before running:** Which count conditions does the string record satisfy, and what additional check can reject it?

**My prediction and reason:** _Write here._


<details>
<summary>How to — Check the datatype</summary>

`parse_shacl_turtle(body)` adds the supplied prefixes and returns a new RDFLib graph. A node shape has `sh:targetClass`; a property shape has `sh:path` and constraints. Construct a complete Turtle fragment for each stage so anonymous property shapes do not accumulate accidentally. The comparison helper supplies repetitive execution; you supply the requirements.

`sh:datatype xsd:date` checks the RDF term's datatype. `Literal("2008-07-18")` is a string; an explicit `datatype=XSD.date` models a date.

</details>


In [ ]:
# S1c — complete your code here.
# Keep earlier results under their own names.


**Observe and explain:** Record the actual result, identify the relevant rule or constraint, and explain your comparison with the prediction.

**My observation and explanation:** _Write here._


<details>
<summary>Show solution — S1c</summary>

```python
s1_shapes = parse_shacl_turtle("""
ex:USReleaseDateShape a sh:NodeShape ;
    sh:targetClass dbo:Film ;
    sh:property [ sh:path ex:releaseDateUSA ; sh:minCount 1 ; sh:maxCount 1 ; sh:datatype xsd:date ] .
""")
s1_representation_results = compare_shacl_cases({"valid": s1_cases["valid"], "string": s1_cases["string"]}, s1_shapes)

assert s1_representation_results["valid"]["conforms"]
assert SH.DatatypeConstraintComponent in s1_representation_results["string"]["components"]
```

The string case has exactly one value, so both counts pass. The datatype finding explains why representation is separate. Validation does not convert date-looking text. The complete profile is now available as `s1_shapes`.

</details>


### S1d — Compare all cases

**Question:** Can each outcome be explained by the complete requirements?

**Use:** `s1_cases` and `s1_shapes` from S1c.

**Task:** Run the supplied comparison pattern and identify focus/path/constraint for each finding.


**Predict before running:** Prepare a four-row prediction/observation table with the expected reason.

**My prediction and reason:** _Write here._


<details>
<summary>How to — Compare all cases</summary>

`compare_shacl_cases(cases, shapes)` returns a case-keyed dictionary. Each record holds conformance, the report, report text and constraint identities. Keep `s1_complete_results` to inspect earlier errors after repairing data.

</details>


In [ ]:
# S1d — supplied loop pattern; uncomment after S1c.
# s1_complete_results = compare_shacl_cases(s1_cases, s1_shapes)
# Record and explain the results.


**Observe and explain:** Record the actual result, identify the relevant rule or constraint, and explain your comparison with the prediction.

**My observation and explanation:** _Write here._


<details>
<summary>Show solution — S1d</summary>

```python
s1_complete_results = compare_shacl_cases(s1_cases, s1_shapes)

s1_expected_components = {
    "valid": set(), "missing": {SH.MinCountConstraintComponent},
    "string": {SH.DatatypeConstraintComponent},
    "two_dates": {SH.MaxCountConstraintComponent},
}
for name, result in s1_complete_results.items():
    assert result["conforms"] == (name == "valid")
    assert result["components"] == s1_expected_components[name]
```

The same film/date path appears in the failing reports, while the component distinguishes absence, representation and excess values. These fixtures isolate one failure each. Other records may violate several conditions.

</details>


### S1e — Repair and revalidate

**Question:** What does each change fix?

**Use:** `s1_cases` and complete `s1_shapes`.

**Task:** Repair every invalid record using the exercise's chosen date and verify the copies.


**Predict before running:** Which values must be added or replaced? Will input fixtures change?

**My prediction and reason:** _Write here._


<details>
<summary>How to — Repair and revalidate</summary>

`copy_graph()` creates an independent graph. `set((subject,predicate,value))` replaces all values for that subject/predicate; `add()` keeps other values. Revalidate with the same shapes so the requirements stay fixed.

</details>


In [ ]:
# S1e — complete your code here.
# Keep earlier results under their own names.


**Observe and explain:** Record the actual result, identify the relevant rule or constraint, and explain your comparison with the prediction.

**My observation and explanation:** _Write here._


<details>
<summary>Show solution — S1e</summary>

```python
s1_repaired_cases = {}
for name in ("missing", "string", "two_dates"):
    repaired = copy_graph(s1_cases[name])
    repaired.set((DBR.The_Dark_Knight, EX.releaseDateUSA,
                  Literal("2008-07-18", datatype=XSD.date)))
    s1_repaired_cases[name] = repaired
s1_repaired_results = compare_shacl_cases(s1_repaired_cases, s1_shapes)
assert all(r["conforms"] for r in s1_repaired_results.values())
assert not list(s1_cases["missing"].objects(DBR.The_Dark_Knight, EX.releaseDateUSA))
```

All repaired rows show True. One typed date fixes absence, a string representation or excess values. The unchanged missing-date fixture demonstrates copying. Passing establishes count/datatype requirements, not independent historical accuracy of the chosen date.

</details>


<details>
<summary>Complete reference solution — S1</summary>

```python
s1_shapes = parse_shacl_turtle("""
ex:USReleaseDateShape a sh:NodeShape ;
    sh:targetClass dbo:Film ;
    sh:property [
        sh:path ex:releaseDateUSA ;
        sh:minCount 1 ;
        sh:maxCount 1 ;
        sh:datatype xsd:date ;
        sh:message "Provide exactly one selected US release date as xsd:date."
    ] .
""")

s1_expected = {
    "valid": None,
    "missing": SH.MinCountConstraintComponent,
    "string": SH.DatatypeConstraintComponent,
    "two_dates": SH.MaxCountConstraintComponent,
}

for name, data in s1_cases.items():
    conforms, report_graph, report_text = validate(
        copy_graph(data), shacl_graph=s1_shapes, inference="none"
    )
    print("\nCASE:", name, "| conforms:", conforms)
    print_shacl_results(report_graph)
    expected_component = s1_expected[name]
    assert bool(conforms) == (expected_component is None)
    if expected_component is not None:
        assert shacl_components(report_graph) == {expected_component}
        assert len(shacl_result_rows(report_graph)) == 1

# In this teaching dataset, we deliberately choose one known correct US date.
for name, data in s1_cases.items():
    repaired = copy_graph(data)
    repaired.set((
        DBR.The_Dark_Knight,
        EX.releaseDateUSA,
        Literal("2008-07-18", datatype=XSD.date),
    ))
    conforms, report_graph, report_text = validate(
        repaired, shacl_graph=s1_shapes, inference="none"
    )
    assert conforms
    print("Repaired:", name, "| conforms:", conforms)

# set() replaced the values in each copy. The original cases are unchanged.
assert len(list(s1_cases["missing"].objects(
    DBR.The_Dark_Knight, EX.releaseDateUSA
))) == 0
```


**Expected results**

| Case | Conforms? | Failed constraint |
|---|---|---|
| valid | True | — |
| missing | False | minimum count |
| string | False | datatype |
| two_dates | False | maximum count |

Each invalid case has exactly one result because it isolates one failed condition. A real record can violate several conditions. The validator does not know which date is factually correct: the repair uses a date chosen explicitly for this exercise.

---

**How to read the code and its output**

In the first loop, each `validate(...)` call returns three values: `conforms` is the overall Boolean, `report_graph` contains the RDF validation results, and `report_text` is their readable text rendering. This solution prints `conforms` and uses `print_shacl_results(report_graph)` to read the RDF report; it does not print `report_text` directly.

The line `CASE: missing | conforms: False` introduces the missing-date case. The helper then prints its focus node, path, value, constraint and message. For all four cases, the focus is `dbr:The_Dark_Knight` and the path is `ex:releaseDateUSA`. The generic message is shared by the constraints, so use the **constraint** field to distinguish their causes:

- `MinCountConstraintComponent`: zero date values; the absent individual `value` is displayed as a dash.
- `DatatypeConstraintComponent`: one value exists, but the reported string `"2008-07-18"` is not an `xsd:date`.
- `MaxCountConstraintComponent`: two distinct dates exceed the allowed count, although both dates have the correct datatype.

For `valid`, the code prints `conforms: True` followed by `No validation results.` The `s1_expected` mapping determines what the assertions check. Passing assertions produce no additional output; a failed one raises `AssertionError`.

**Reading the repair results**

The second loop prints `Repaired: ... | conforms: True` for every case. Its `set()` call replaces all date values in a copy with one typed date. The last assertion silently confirms that the original missing-date fixture is unchanged.

The variables are reassigned on every validation call. After both loops, `conforms` and `report_graph` describe the **last repaired case**, not any earlier failing record.

</details>


## S2 — When is a number acceptable?

**What is this about?** A correct datatype does not guarantee a suitable value. Required and optional properties add a different completeness question.

**After this exercise, you can:**

- Distinguish datatype and numeric-bound failures.
- Explain optional versus required fields.
- Test repairs and boundaries.

**Given:** `s2_baseline` and `s2_cases`.

**Your result:** A numeric shape, labelled reports and boundary experiments.

**Application requirements:** For every targeted `dbo:Film`, require exactly one `ex:runtimeMinutes` value: an `xsd:integer` greater than 0. `ex:receptionScore` is optional, has at most one value, and, if present, must be an `xsd:integer` between 0 and 100 inclusive.

Work through the lettered parts in order. Rerun the supplied preparation cell to reset this exercise. Earlier exercises' student answers are not required.


<details>
<summary>Original complete task description — S2</summary>

## Exercise S2 — validate numbers and value ranges

**Goal:** check typed values and distinguish required from optional properties.

Extend your SHACL skills with a new, independent shape for `dbo:Film`:

- **Runtime:** `ex:runtimeMinutes` is required, has exactly one value, is an `xsd:integer`, and is greater than 0.
- **Reception score:** `ex:receptionScore` is optional, has at most one value, and if present is an `xsd:integer` between 0 and 100 inclusive.

**Tasks**

1. Predict the outcomes for the cases below.
2. Write `s2_shapes`, validate all cases, and explain the reported constraint.
3. Repair the three deliberately invalid values on fresh copies.
4. Explain why the record with no reception score can still conform.

Use `sh:minExclusive`, `sh:minInclusive`, and `sh:maxInclusive` for the ranges. We use an explicitly named minutes property because DBpedia's `dbo:runtime` denotes seconds.

</details>


In [ ]:
s2_baseline = parse_shacl_turtle("""
dbr:The_Dark_Knight a dbo:Film ;
    ex:runtimeMinutes 152 ;
    ex:receptionScore 94 .
""")

s2_cases = {"valid": copy_graph(s2_baseline)}

s2_cases["runtime_as_string"] = copy_graph(s2_baseline)
s2_cases["runtime_as_string"].set((
    DBR.The_Dark_Knight, EX.runtimeMinutes, Literal("152")
))

s2_cases["negative_runtime"] = copy_graph(s2_baseline)
s2_cases["negative_runtime"].set((
    DBR.The_Dark_Knight, EX.runtimeMinutes, Literal(-10)
))

s2_cases["score_too_large"] = copy_graph(s2_baseline)
s2_cases["score_too_large"].set((
    DBR.The_Dark_Knight, EX.receptionScore, Literal(140)
))

s2_cases["no_optional_score"] = copy_graph(s2_baseline)
s2_cases["no_optional_score"].remove((
    DBR.The_Dark_Knight, EX.receptionScore, None
))


### S2a — Model numeric requirements

**Question:** Which conditions express presence, datatype and bounds?

**Use:** `s2_cases`; independent of S1.

**Task:** Build `s2_shapes` for all stated requirements and validate the baseline.


**Predict before running:** Which condition requires runtime, and why may a score be absent?

**My prediction and reason:** _Write here._


<details>
<summary>How to — Model numeric requirements</summary>

`sh:minExclusive` excludes its bound; `sh:minInclusive` and `sh:maxInclusive` include theirs. Datatype/range checks examine present values. Presence needs minimum count.

</details>


In [ ]:
# S2a — complete your code here.
# Keep earlier results under their own names.


**Observe and explain:** Record the actual result, identify the relevant rule or constraint, and explain your comparison with the prediction.

**My observation and explanation:** _Write here._


<details>
<summary>Show solution — S2a</summary>

```python
s2_shapes = parse_shacl_turtle("""
ex:NumericFilmShape a sh:NodeShape ;
    sh:targetClass dbo:Film ;
    sh:property [
        sh:path ex:runtimeMinutes ;
        sh:minCount 1 ;
        sh:maxCount 1 ;
        sh:datatype xsd:integer ;
        sh:minExclusive 0
    ] ;
    sh:property [
        sh:path ex:receptionScore ;
        sh:maxCount 1 ;
        sh:datatype xsd:integer ;
        sh:minInclusive 0 ;
        sh:maxInclusive 100
    ] .
""")
s2_baseline_results = compare_shacl_cases({"valid": s2_cases["valid"]}, s2_shapes)

assert s2_baseline_results["valid"]["conforms"]
```

The baseline passes because 152 is a positive integer and 94 lies inside the score bounds. One successful record is not enough to establish that intended errors are rejected; the next step challenges the requirements.

</details>


### S2b — Compare invalid and optional values

**Question:** Which requirement fails for each controlled change?

**Use:** `s2_cases`, `s2_baseline`, `s2_shapes` from S2a.

**Task:** Compare all cases, plus a missing-runtime copy, preserving the reports.


**Predict before running:** Predict each provided case and a new copy with the required runtime removed.

**My prediction and reason:** _Write here._


<details>
<summary>How to — Compare invalid and optional values</summary>

`shacl_components(report)` extracts component identities. Membership checks require a relevant finding without assuming exactly one. Read the focus/path/value/component fields together.

</details>


In [ ]:
# S2b — complete your code here.
# Keep earlier results under their own names.


**Observe and explain:** Record the actual result, identify the relevant rule or constraint, and explain your comparison with the prediction.

**My observation and explanation:** _Write here._


<details>
<summary>Show solution — S2b</summary>

```python
s2_experiments = dict(s2_cases)
s2_missing_runtime = copy_graph(s2_baseline)
s2_missing_runtime.remove((DBR.The_Dark_Knight, EX.runtimeMinutes, None))
s2_experiments["missing_required_runtime"] = s2_missing_runtime
s2_validation_results = compare_shacl_cases(s2_experiments, s2_shapes)
assert s2_validation_results["no_optional_score"]["conforms"]
assert SH.MinCountConstraintComponent in s2_validation_results["missing_required_runtime"]["components"]
assert SH.DatatypeConstraintComponent in s2_validation_results["runtime_as_string"]["components"]
assert SH.MinExclusiveConstraintComponent in s2_validation_results["negative_runtime"]["components"]
assert SH.MaxInclusiveConstraintComponent in s2_validation_results["score_too_large"]["components"]
```

The string runtime has both datatype and numeric-comparison findings in this example. A negative integer has the right datatype but fails its bound. Missing optional score passes; missing required runtime fails minimum count. Reports remain available in `s2_validation_results`.

</details>


### S2c — Repair and examine boundaries

**Question:** How do inclusive and exclusive limits differ?

**Use:** `s2_cases`, `s2_baseline`, `s2_shapes`.

**Task:** Repair the three invalid values. Then validate a runtime of 0, a score of 0, and a score of 100, each in an independent baseline copy.


**Predict before running:** Predict runtime 0 and scores 0 and 100.

**My prediction and reason:** _Write here._


<details>
<summary>How to — Repair and examine boundaries</summary>

`Literal(152)` is an integer; `Literal("152")` is a string. Use independent copies and `set()` to replace one field. Keep repair and boundary results separately.

</details>


In [ ]:
# S2c — complete your code here.
# Keep earlier results under their own names.


**Observe and explain:** Record the actual result, identify the relevant rule or constraint, and explain your comparison with the prediction.

**My observation and explanation:** _Write here._


<details>
<summary>Show solution — S2c</summary>

```python
s2_repaired_cases = {}
for name in ("runtime_as_string", "negative_runtime", "score_too_large"):
    fixed = copy_graph(s2_cases[name])
    fixed.set((DBR.The_Dark_Knight, EX.runtimeMinutes, Literal(152)))
    fixed.set((DBR.The_Dark_Knight, EX.receptionScore, Literal(94)))
    s2_repaired_cases[name] = fixed
s2_repaired_results = compare_shacl_cases(s2_repaired_cases, s2_shapes)
assert all(r["conforms"] for r in s2_repaired_results.values())
s2_boundaries = {}
for name, predicate, value in (
    ("runtime_zero", EX.runtimeMinutes, 0),
    ("score_zero", EX.receptionScore, 0),
    ("score_hundred", EX.receptionScore, 100),
):
    data = copy_graph(s2_baseline)
    data.set((DBR.The_Dark_Knight, predicate, Literal(value)))
    s2_boundaries[name] = data
s2_boundary_results = compare_shacl_cases(s2_boundaries, s2_shapes)
assert not s2_boundary_results["runtime_zero"]["conforms"]
assert s2_boundary_results["score_zero"]["conforms"]
assert s2_boundary_results["score_hundred"]["conforms"]
```

The repairs pass. Runtime 0 fails its exclusive bound, while scores 0 and 100 pass their inclusive bounds. The tables supply visible evidence; assertions supplement them silently. The validator checks chosen replacements rather than calculating film facts.

</details>


<details>
<summary>Complete reference solution — S2</summary>

```python
s2_shapes = parse_shacl_turtle("""
ex:NumericFilmShape a sh:NodeShape ;
    sh:targetClass dbo:Film ;
    sh:property [
        sh:path ex:runtimeMinutes ;
        sh:minCount 1 ;
        sh:maxCount 1 ;
        sh:datatype xsd:integer ;
        sh:minExclusive 0
    ] ;
    sh:property [
        sh:path ex:receptionScore ;
        sh:maxCount 1 ;
        sh:datatype xsd:integer ;
        sh:minInclusive 0 ;
        sh:maxInclusive 100
    ] .
""")

for name, data in s2_cases.items():
    conforms, report_graph, report_text = validate(
        copy_graph(data), shacl_graph=s2_shapes, inference="none"
    )
    print("\nCASE:", name, "| conforms:", conforms)
    print_shacl_results(report_graph)
    assert bool(conforms) == (name in {"valid", "no_optional_score"})
    if name == "runtime_as_string":
        assert SH.DatatypeConstraintComponent in shacl_components(report_graph)
    if name == "negative_runtime":
        assert SH.MinExclusiveConstraintComponent in shacl_components(report_graph)
    if name == "score_too_large":
        assert SH.MaxInclusiveConstraintComponent in shacl_components(report_graph)

for name in ("runtime_as_string", "negative_runtime", "score_too_large"):
    repaired = copy_graph(s2_cases[name])
    repaired.set((DBR.The_Dark_Knight, EX.runtimeMinutes, Literal(152)))
    repaired.set((DBR.The_Dark_Knight, EX.receptionScore, Literal(94)))
    conforms, report_graph, report_text = validate(
        repaired, shacl_graph=s2_shapes, inference="none"
    )
    assert conforms
    print("Repaired:", name, "| conforms:", conforms)
```


The runtime is required by `sh:minCount 1`. The score has no minimum count, so absence is allowed. Datatype and range constraints apply to values that are present. The string-valued runtime can produce more than one result because the value also fails the numeric comparison; one input error need not correspond to exactly one report entry.

---

**How to read the code and its output**

The first loop prints a `CASE` heading and its `conforms` Boolean, then `print_shacl_results(report_graph)` shows the individual findings. Read **focus node → path → value → constraint** together: which film, which field, which supplied value, and which rule failed?

| Case | Printed result and interpretation |
|---|---|
| `valid` | `True` and no validation results: 152 and 94 satisfy their rules. |
| `runtime_as_string` | `False`; the value `"152"` appears in datatype and minimum-exclusive findings on `ex:runtimeMinutes`. |
| `negative_runtime` | `False`; integer −10 fails `MinExclusiveConstraintComponent` on the runtime path. |
| `score_too_large` | `False`; integer 140 fails `MaxInclusiveConstraintComponent` on `ex:receptionScore`. |
| `no_optional_score` | `True` and no findings: absence is permitted for the score. |

The two string-runtime findings refer to **one supplied value**. Text containing digits does not satisfy the integer datatype or the numeric greater-than comparison. The message field explains the particular failed condition.

`shacl_components(report_graph)` extracts the component identities into a set. The assertions use membership tests, so they require the relevant component without assuming that each case produces exactly one finding. Successful assertions print nothing.

**Reading the repair and boundary results**

The repair loop prints `Repaired: ... | conforms: True` after replacing the values with integers 152 and 94 in copies. It supplies values; SHACL does not calculate them.

Runtime 0 would fail because its lower bound is exclusive. Scores 0 and 100 are allowed because both score bounds are inclusive. A missing runtime would fail `minCount 1`; the score's `maxCount 1` still allows zero values.

After the repair loop, `conforms` and `report_graph` hold the **last repaired result**. To inspect an earlier error, read its printed block or inspect its report during the first loop.

</details>


## S3 — What makes a director reference usable?

**What is this about?** A name string, an IRI and a classified IRI carry different information.

**After this exercise, you can:**

- Distinguish relationship presence, node kind and class evidence.
- Identify focus node versus checked value.
- Explain at-least-one cardinality.

**Given:** Four independent graphs in `s3_cases`.

**Your result:** Partial/complete shapes, preserved comparison reports and verified repairs.

**Application requirements:** For every targeted film, require at least one `dbo:director` value. Every such value must be an IRI recognized as an instance of `dbo:Director`. Multiple directors remain allowed; the first stage deliberately omits the class requirement.

Work through the lettered parts in order. Rerun the supplied preparation cell to reset this exercise. Earlier exercises' student answers are not required.


<details>
<summary>Original complete task description — S3</summary>

## Exercise S3 — validate relationships, node kinds, and classes

**Goal:** distinguish a missing relationship, a text value, and an inadequately classified resource.

For the exercise application, each film must have **at least one** `dbo:director` value. Every value must be an IRI and an instance of `dbo:Director`.

**Tasks**

1. Write `s3_shapes` using `sh:minCount`, `sh:nodeKind sh:IRI`, and `sh:class dbo:Director`.
2. Validate all four independent cases below, initially with `inference="none"`.
3. Repair copies of the invalid cases, preserving a relationship to a director resource rather than replacing it with a name string.
4. Add a second correctly typed director to the valid record. Should it pass?

There is **no maximum count of 1**: multiple directors are possible. The required relationship and type information are rules of this exercise, not a completeness claim about the live DBpedia dataset.

</details>


In [ ]:
s3_cases = {
    "valid": parse_shacl_turtle("""
        dbr:Interstellar a dbo:Film ;
            dbo:director dbr:Christopher_Nolan .
        dbr:Christopher_Nolan a dbo:Director .
    """),
    "missing_director": parse_shacl_turtle("""
        dbr:Interstellar a dbo:Film .
    """),
    "literal_director": parse_shacl_turtle("""
        dbr:Interstellar a dbo:Film ;
            dbo:director "Christopher Nolan" .
    """),
    "untyped_director": parse_shacl_turtle("""
        dbr:Interstellar a dbo:Film ;
            dbo:director dbr:Christopher_Nolan .
    """),
}


### S3a — Require a reference

**Question:** Does the record supply a relationship to an IRI?

**Use:** `s3_cases`; no earlier solution.

**Task:** Create partial `s3_reference_shapes` without a class constraint and compare the records.


**Predict before running:** Compare missing, literal and untyped-IRI records using presence and node kind only.

**My prediction and reason:** _Write here._


<details>
<summary>How to — Require a reference</summary>

`sh:minCount` checks presence. `sh:nodeKind sh:IRI` checks the value's RDF kind; it does not establish class membership.

</details>


In [ ]:
# S3a — complete your code here.
# Keep earlier results under their own names.


**Observe and explain:** Record the actual result, identify the relevant rule or constraint, and explain your comparison with the prediction.

**My observation and explanation:** _Write here._


<details>
<summary>Show solution — S3a</summary>

```python
s3_reference_shapes = parse_shacl_turtle("""
ex:DirectedFilmShape a sh:NodeShape ;
    sh:targetClass dbo:Film ;
    sh:property [
        sh:path dbo:director ;
        sh:minCount 1 ;
        sh:nodeKind sh:IRI
    ] .
""")
s3_reference_results = compare_shacl_cases(s3_cases, s3_reference_shapes)

assert not s3_reference_results["missing_director"]["conforms"]
assert not s3_reference_results["literal_director"]["conforms"]
assert s3_reference_results["untyped_director"]["conforms"]
```

The untyped IRI passes this partial profile because it is a present IRI. The literal fails node kind; the absent relationship fails minimum count. Classification has not yet been required.

</details>


### S3b — Require class evidence

**Question:** What changes when the object must be a Director?

**Use:** `s3_cases`; previous comparison `s3_reference_results`.

**Task:** Build the complete `s3_shapes` and compare the two profiles.


**Predict before running:** Which case changes outcome? Can a single value fail two conditions?

**My prediction and reason:** _Write here._


<details>
<summary>How to — Require class evidence</summary>

`sh:class` checks recognized instance membership in the effective data graph, including relevant subclass paths. Additional RDFS expansion remains disabled here.

</details>


In [ ]:
# S3b — complete your code here.
# Keep earlier results under their own names.


**Observe and explain:** Record the actual result, identify the relevant rule or constraint, and explain your comparison with the prediction.

**My observation and explanation:** _Write here._


<details>
<summary>Show solution — S3b</summary>

```python
s3_shapes = parse_shacl_turtle("""
ex:DirectedFilmShape a sh:NodeShape ;
    sh:targetClass dbo:Film ;
    sh:property [
        sh:path dbo:director ;
        sh:minCount 1 ;
        sh:nodeKind sh:IRI ;
        sh:class dbo:Director
    ] .
""")
s3_class_results = compare_shacl_cases(s3_cases, s3_shapes)

display_rows([
    {"case": name, "IRI profile": s3_reference_results[name]["conforms"],
     "with class requirement": s3_class_results[name]["conforms"]}
    for name in s3_cases
])
assert not s3_class_results["untyped_director"]["conforms"]
assert s3_class_results["literal_director"]["components"] == {
    SH.NodeKindConstraintComponent, SH.ClassConstraintComponent
}
```

The untyped IRI now fails class membership; the literal fails both node kind and class. Interstellar remains the focus node. The director object is the checked value, which can appear in the report. No external descriptions are fetched automatically.

</details>


### S3c — Repair representation and classification

**Question:** Which edits address the distinct failures?

**Use:** `s3_cases`, complete `s3_shapes` from S3b.

**Task:** Repair fresh copies, keeping the complete requirements fixed.


**Predict before running:** Would typing the Nolan IRI alone replace a literal-valued relationship?

**My prediction and reason:** _Write here._


<details>
<summary>How to — Repair representation and classification</summary>

`set()` replaces relationship objects; `add((director, RDF.type, DBO.Director))` supplies class evidence. These edit different RDF triples.

</details>


In [ ]:
# S3c — complete your code here.
# Keep earlier results under their own names.


**Observe and explain:** Record the actual result, identify the relevant rule or constraint, and explain your comparison with the prediction.

**My observation and explanation:** _Write here._


<details>
<summary>Show solution — S3c</summary>

```python
s3_repaired_cases = {}
for name, data in s3_cases.items():
    fixed = copy_graph(data)
    fixed.set((DBR.Interstellar, DBO.director, DBR.Christopher_Nolan))
    fixed.add((DBR.Christopher_Nolan, RDF.type, DBO.Director))
    s3_repaired_cases[name] = fixed
s3_repaired_results = compare_shacl_cases(s3_repaired_cases, s3_shapes)
assert all(r["conforms"] for r in s3_repaired_results.values())
```

Every repaired row passes. A reference edit fixes the absent/literal relationship, and the added type fixes missing classification. Typing a separate IRI by itself would leave a literal object unchanged.

</details>


### S3d — Test another director

**Question:** Does at least one mean exactly one?

**Use:** `s3_cases["valid"]`, `s3_shapes`; the added director is synthetic.

**Task:** Add a second correctly typed director to a copy and validate.


**Predict before running:** Predict the second valid director's effect.

**My prediction and reason:** _Write here._


<details>
<summary>How to — Test another director</summary>

`add()` preserves the first director. Every value must satisfy the IRI and class checks. A maximum count would be a separate constraint.

</details>


In [ ]:
# S3d — complete your code here.
# Keep earlier results under their own names.


**Observe and explain:** Record the actual result, identify the relevant rule or constraint, and explain your comparison with the prediction.

**My observation and explanation:** _Write here._


<details>
<summary>Show solution — S3d</summary>

```python
s3_multiple_data = copy_graph(s3_cases["valid"])
s3_multiple_data.add((DBR.Interstellar, DBO.director, EX.SecondDirector))
s3_multiple_data.add((EX.SecondDirector, RDF.type, DBO.Director))
s3_multiple_conforms, s3_multiple_report, _ = validate(
    s3_multiple_data, shacl_graph=s3_shapes, inference="none"
)
display_rows([{"director values": len(list(s3_multiple_data.objects(DBR.Interstellar, DBO.director))),
               "conforms": bool(s3_multiple_conforms)}])
print_shacl_results(s3_multiple_report)
assert s3_multiple_conforms
```

The table shows two values and True: both values meet the constraints, and no maximum count is stated. This is an experiment about the profile, not a factual claim about the film.

</details>


<details>
<summary>Complete reference solution — S3</summary>

```python
s3_shapes = parse_shacl_turtle("""
ex:DirectedFilmShape a sh:NodeShape ;
    sh:targetClass dbo:Film ;
    sh:property [
        sh:path dbo:director ;
        sh:minCount 1 ;
        sh:nodeKind sh:IRI ;
        sh:class dbo:Director
    ] .
""")

s3_expected = {
    "valid": set(),
    "missing_director": {SH.MinCountConstraintComponent},
    "literal_director": {
        SH.NodeKindConstraintComponent, SH.ClassConstraintComponent
    },
    "untyped_director": {SH.ClassConstraintComponent},
}

for name, data in s3_cases.items():
    conforms, report_graph, report_text = validate(
        copy_graph(data), shacl_graph=s3_shapes, inference="none"
    )
    print("\nCASE:", name, "| conforms:", conforms)
    print_shacl_results(report_graph)
    assert bool(conforms) == (name == "valid")
    assert shacl_components(report_graph) == s3_expected[name]

for name, data in s3_cases.items():
    repaired = copy_graph(data)
    repaired.set((DBR.Interstellar, DBO.director, DBR.Christopher_Nolan))
    repaired.add((DBR.Christopher_Nolan, RDF.type, DBO.Director))
    conforms, report_graph, report_text = validate(
        repaired, shacl_graph=s3_shapes, inference="none"
    )
    assert conforms

# A synthetic second director illustrates cardinality, not a claim about this film.
s3_two_directors = copy_graph(s3_cases["valid"])
s3_two_directors.add((DBR.Interstellar, DBO.director, EX.SecondDirector))
s3_two_directors.add((EX.SecondDirector, RDF.type, DBO.Director))
conforms, report_graph, report_text = validate(
    s3_two_directors, shacl_graph=s3_shapes, inference="none"
)
print("Two classified director resources conform:", conforms)
assert conforms
```


The literal `"Christopher Nolan"` violates both the IRI and class constraints, so two results are appropriate. The untyped IRI already has the right node kind; only the class condition fails. Repairing missing local type information does not retrieve information from DBpedia automatically.

---

**How to read the code and its output**

`s3_expected` maps each case name to a **set of expected constraint components**. These are rule identities, not report-message text or a prescribed message order. The first loop prints `CASE: ... | conforms: ...`, displays the findings, and compares their component set with the mapping. Passing assertions remain silent.

| Case | What the printed report establishes |
|---|---|
| `valid` | `True`, with no findings: the relationship exists and its IRI object is classified as a director. |
| `missing_director` | `False` with `MinCountConstraintComponent`: there is no relationship value. |
| `literal_director` | `False` with node-kind and class components: a name string satisfies neither condition. |
| `untyped_director` | `False` with the class component only: the IRI condition already passes. |

The **focus node** is `dbr:Interstellar`, and the **path** is `dbo:director`. A reported `value` such as `dbr:Christopher_Nolan` is the object being checked, not the film targeted by the shape. The missing case has no individual value to report; its problem is a count of zero.

**Reading the two repair checks**

The repair loop uses `set()` to supply the IRI relationship and `add()` to supply its type. This loop contains only `assert conforms`, so successful repairs produce **no printed confirmation**.

The final experiment explicitly prints `Two classified director resources conform: True`. Both director IRIs satisfy the value constraints, and there is no maximum count. “At least one” therefore does not mean “exactly one”; the synthetic example illustrates that rule without making a factual claim about a second director.

At the end, `conforms` and `report_graph` describe this **two-director experiment**, because each later validation overwrites the earlier values.

</details>


## S5 — Which languages are permitted or required?

**What is this about?** Language tags distinguish RDF values. Permitting a language differs from requiring its presence.

**After this exercise, you can:**

- Explain language admissibility and uniqueness.
- Distinguish identical triples from distinct values.
- Explain allowed versus required coverage.

**Given:** `s5_baseline` and `s5_cases`.

**Your result:** Language profiles, comparison reports and an English-only counterexample.

**Application requirements:** For each targeted film, require at least one `rdfs:label`. Every label must match the allowed language ranges `de`, `en`, or `uk`, and there must be at most one value per nonempty language tag. No individual language is required by this basic profile.

Work through the lettered parts in order. Rerun the supplied preparation cell to reset this exercise. Earlier exercises' student answers are not required.


<details>
<summary>Original complete task description — S5</summary>

## Bonus exercise S5 — multilingual labels

**Goal:** validate language-tagged literals and distinguish “allowed” from “required”.

For this application's preferred film labels:

- allow `de`, `en`, and `uk` using `sh:languageIn`;
- allow at most one value per language tag using `sh:uniqueLang true`;
- require at least one label overall.

**Tasks**

1. Write `s5_shapes` and validate the cases below.
2. Explain the problems with a plain literal, a French label, and two **different** German labels.
3. Add the same German triple twice to a valid graph. Does that produce two values?
4. Repair fresh copies of the invalid graphs.
5. Does `sh:languageIn ("de" "en" "uk")` require all three languages to be present?

Use `uk` for Ukrainian; `UA` is a country code. RDF list syntax such as `("de" "en" "uk")` is used in the shapes graph.

</details>


In [ ]:
s5_baseline = parse_shacl_turtle("""
dbr:The_Dark_Knight a dbo:Film ;
    rdfs:label "The Dark Knight"@en,
               "The Dark Knight"@de,
               "Темний лицар"@uk .
""")
s5_cases = {"valid": copy_graph(s5_baseline)}

s5_cases["plain_literal"] = copy_graph(s5_baseline)
s5_cases["plain_literal"].add((
    DBR.The_Dark_Knight, RDFS.label, Literal("The Dark Knight")
))

s5_cases["not_allowed_language"] = copy_graph(s5_baseline)
s5_cases["not_allowed_language"].add((
    DBR.The_Dark_Knight, RDFS.label, Literal("Le Chevalier noir", lang="fr")
))

s5_cases["two_distinct_german_labels"] = copy_graph(s5_baseline)
s5_cases["two_distinct_german_labels"].add((
    DBR.The_Dark_Knight, RDFS.label, Literal("Der dunkle Ritter", lang="de")
))


### S5a — Check language admissibility

**Question:** Does every label use a permitted language?

**Use:** `s5_cases`; independent preparation.

**Task:** Build partial `s5_allowed_shapes` without uniqueness and compare all cases.


**Predict before running:** Predict plain, French and two-German-label cases without a uniqueness condition.

**My prediction and reason:** _Write here._


<details>
<summary>How to — Check language admissibility</summary>

`sh:languageIn` takes an RDF list of language ranges and checks every present value. It does not require every listed language. These fixtures use simple tags; ranges can also match corresponding subtags.

**RDF-list syntax example**

Inside a property shape, a neutral example of the language-list parameter is:

```turtle
sh:languageIn ("fr" "es")
```

The parentheses make an RDF list; the entries are ordinary string literals representing language ranges. Replace these example ranges with the ranges required by your application. This syntax example does not specify which languages your film exercise should allow.

</details>


In [ ]:
# S5a — complete your code here.
# Keep earlier results under their own names.


**Observe and explain:** Record the actual result, identify the relevant rule or constraint, and explain your comparison with the prediction.

**My observation and explanation:** _Write here._


<details>
<summary>Show solution — S5a</summary>

```python
s5_allowed_shapes = parse_shacl_turtle("""
ex:FilmLabelShape a sh:NodeShape ;
    sh:targetClass dbo:Film ;
    sh:property [
        sh:path rdfs:label ;
        sh:minCount 1 ;
        sh:languageIn ("de" "en" "uk")
    ] .
""")
s5_allowed_results = compare_shacl_cases(s5_cases, s5_allowed_shapes)

assert not s5_allowed_results["plain_literal"]["conforms"]
assert not s5_allowed_results["not_allowed_language"]["conforms"]
assert s5_allowed_results["two_distinct_german_labels"]["conforms"]
```

The plain and French labels fail admissibility. Both German labels pass this partial profile, since uniqueness has not yet been required. Existing valid values cannot compensate for an additional invalid label.

</details>


### S5b — Investigate uniqueness and identical triples

**Question:** What counts as another label value?

**Use:** `s5_cases`, `s5_baseline` and prior profile.

**Task:** Build complete `s5_shapes`, compare the cases and separately re-add the existing German triple.


**Predict before running:** Will two German texts fail? Will re-adding an existing German triple change graph size?

**My prediction and reason:** _Write here._


<details>
<summary>How to — Investigate uniqueness and identical triples</summary>

`sh:uniqueLang true` rejects distinct values sharing a nonempty language tag. RDF graphs are sets: adding the same triple again adds no second value.

</details>


In [ ]:
# S5b — complete your code here.
# Keep earlier results under their own names.


**Observe and explain:** Record the actual result, identify the relevant rule or constraint, and explain your comparison with the prediction.

**My observation and explanation:** _Write here._


<details>
<summary>Show solution — S5b</summary>

```python
s5_shapes = parse_shacl_turtle("""
ex:FilmLabelShape a sh:NodeShape ;
    sh:targetClass dbo:Film ;
    sh:property [
        sh:path rdfs:label ;
        sh:minCount 1 ;
        sh:languageIn ("de" "en" "uk") ;
        sh:uniqueLang true
    ] .
""")
s5_unique_results = compare_shacl_cases(s5_cases, s5_shapes)

assert SH.UniqueLangConstraintComponent in s5_unique_results["two_distinct_german_labels"]["components"]
s5_duplicate_data = copy_graph(s5_baseline)
s5_count_before = len(s5_duplicate_data)
s5_duplicate_data.add((DBR.The_Dark_Knight, RDFS.label, Literal("The Dark Knight", lang="de")))
s5_duplicate_conforms, s5_duplicate_report, _ = validate(
    s5_duplicate_data, shacl_graph=s5_shapes, inference="none"
)
display_rows([{"triples before": s5_count_before, "triples after": len(s5_duplicate_data),
               "conforms": bool(s5_duplicate_conforms)}])
assert len(s5_duplicate_data) == s5_count_before
assert s5_duplicate_conforms
```

Two different German values fail uniqueness. Re-adding the same triple leaves size and conformance unchanged. English and German labels with equal text are different RDF terms because their tags differ.

</details>


### S5c — Repair and test English-only coverage

**Question:** Does permitting three languages require all three?

**Use:** `s5_cases`, `s5_baseline`, complete `s5_shapes`.

**Task:** Restore the supplied preferred labels in copies and validate an English-only graph.


**Predict before running:** Predict a record with one English label.

**My prediction and reason:** _Write here._


<details>
<summary>How to — Repair and test English-only coverage</summary>

Use `remove((film, RDFS.label, None))` on a copy, then add chosen replacements. Admissibility and language-specific coverage are separate requirements.

</details>


In [ ]:
# S5c — complete your code here.
# Keep earlier results under their own names.


**Observe and explain:** Record the actual result, identify the relevant rule or constraint, and explain your comparison with the prediction.

**My observation and explanation:** _Write here._


<details>
<summary>Show solution — S5c</summary>

```python
s5_repaired_cases = {}
for name, data in s5_cases.items():
    fixed = copy_graph(data)
    fixed.remove((DBR.The_Dark_Knight, RDFS.label, None))
    for label in s5_baseline.objects(DBR.The_Dark_Knight, RDFS.label):
        fixed.add((DBR.The_Dark_Knight, RDFS.label, label))
    s5_repaired_cases[name] = fixed
s5_repaired_results = compare_shacl_cases(s5_repaired_cases, s5_shapes)
assert all(r["conforms"] for r in s5_repaired_results.values())
s5_english_data = parse_shacl_turtle("""
dbr:The_Dark_Knight a dbo:Film ; rdfs:label "The Dark Knight"@en .
""")
s5_english_conforms, s5_english_report, _ = validate(
    s5_english_data, shacl_graph=s5_shapes, inference="none"
)
display_rows([{"record": "English only", "conforms": bool(s5_english_conforms)}])
print_shacl_results(s5_english_report)
assert s5_english_conforms
```

The restored records pass. English-only also passes: at least one label exists, its language is allowed and no tag repeats. The next extension adds required coverage. Restoring baseline labels is an explicit exercise choice, not automatic translation.

</details>


<details>
<summary>Complete reference solution — S5</summary>

```python
s5_shapes = parse_shacl_turtle("""
ex:FilmLabelShape a sh:NodeShape ;
    sh:targetClass dbo:Film ;
    sh:property [
        sh:path rdfs:label ;
        sh:minCount 1 ;
        sh:languageIn ("de" "en" "uk") ;
        sh:uniqueLang true
    ] .
""")
s5_expected = {
    "valid": set(),
    "plain_literal": {SH.LanguageInConstraintComponent},
    "not_allowed_language": {SH.LanguageInConstraintComponent},
    "two_distinct_german_labels": {SH.UniqueLangConstraintComponent},
}

for name, data in s5_cases.items():
    conforms, report_graph, report_text = validate(
        copy_graph(data), shacl_graph=s5_shapes, inference="none"
    )
    print("\nCASE:", name, "| conforms:", conforms)
    print_shacl_results(report_graph)
    assert bool(conforms) == (name == "valid")
    assert shacl_components(report_graph) == s5_expected[name]

# Re-adding the same RDF triple does not create a second value.
same_triple_twice = copy_graph(s5_baseline)
count_before = len(same_triple_twice)
same_triple_twice.add((
    DBR.The_Dark_Knight, RDFS.label, Literal("The Dark Knight", lang="de")
))
assert len(same_triple_twice) == count_before
conforms, report_graph, _ = validate(
    same_triple_twice, shacl_graph=s5_shapes, inference="none"
)
assert conforms

# Explicit repairs chosen for these synthetic error cases.
for name, data in s5_cases.items():
    repaired = copy_graph(data)
    repaired.remove((DBR.The_Dark_Knight, RDFS.label, None))
    for label in s5_baseline.objects(DBR.The_Dark_Knight, RDFS.label):
        repaired.add((DBR.The_Dark_Knight, RDFS.label, label))
    conforms, report_graph, _ = validate(
        repaired, shacl_graph=s5_shapes, inference="none"
    )
    assert conforms

# One allowed language is enough for the shape above.
only_english = parse_shacl_turtle("""
dbr:The_Dark_Knight a dbo:Film ; rdfs:label "The Dark Knight"@en .
""")
conforms, report_graph, _ = validate(
    only_english, shacl_graph=s5_shapes, inference="none"
)
print("Only English, but all present labels satisfy the rules:", conforms)
assert conforms
```


`sh:languageIn` restricts each label's language; it does not require every listed language. `sh:uniqueLang` rejects two distinct values sharing a nonempty language tag. Identical triples cannot create duplicate RDF values.

---

**How to read the code and its output**

The first loop prints each `CASE` name, its `conforms` Boolean, and the report fields. The focus is `dbr:The_Dark_Knight` and the path is `rdfs:label`. `s5_expected` specifies component sets; equality with `shacl_components(report_graph)` checks which rules failed without depending on message order. Successful assertions print nothing.

The baseline's three labels produce `True` and no findings. The English and German text may be identical, but their language tags make them different RDF values. In the failing cases:

- `plain_literal` reports `LanguageInConstraintComponent` for a label with no matching language tag.
- `not_allowed_language` reports that component for the French value `"Le Chevalier noir"@fr`.
- `two_distinct_german_labels` reports `UniqueLangConstraintComponent`: two different values use `de`.

Existing valid labels do not cancel an invalid additional value. The language condition applies to every label; uniqueness concerns repeated tags among distinct label values.

**Reading the silent and printed follow-up checks**

The `same_triple_twice` checks silently confirm that re-adding the existing German triple changes neither graph size nor conformance. An RDF graph stores a set of triples, so the operation has not created a second German value.

The repair loop also contains only assertions and prints nothing when successful. It restores the supplied baseline labels in each copy; SHACL has not selected or translated labels automatically.

The final print is `Only English, but all present labels satisfy the rules: True`. That graph has one allowed label, meeting the overall minimum, with no repeated tag. Allowed languages are not required languages.

Because validation variables are reused, `conforms` and `report_graph` now describe this **English-only graph**. Inspect the earlier printed case blocks for the original error results.

</details>


## S4 — Which records does a validation result cover?

**What is this about?** Target selection and requirements are separate. Inference can change selection or class evidence.

**After this exercise, you can:**

- Explain class-based and relationship-based targets.
- Compare configurations using preserved reports.
- Distinguish unchecked records from checked conforming records.

**Given:** `s4_data` and `s4_schema`.

**Your result:** Four named reports and an explanation of selection versus conformance.

**Application requirements:** Initially target resources recognized as `dbo:Film` and require at least one `ex:releaseDateUSA` value. This experiment checks presence only; it does not impose S1's date datatype or maximum count. Compare the class target with `sh:targetSubjectsOf dbo:director` while keeping that presence requirement.

Work through the lettered parts in order. Rerun the supplied preparation cell to reset this exercise. Earlier exercises' student answers are not required.


<details>
<summary>Original complete task description — S4</summary>

## Exercise S4 — what does the validator actually check?

**Goal:** understand targets and the effect of RDFS inference.

An imported record has a `dbo:director` relationship but no explicit film type and no US release date. The local schema declares that subjects of `dbo:director` are films.

**Tasks**

1. Write `s4_shapes`: target `dbo:Film` and require an `ex:releaseDateUSA` value. This exercise only needs `sh:minCount 1`.
2. Predict and compare validation with `inference="none"` and `inference="rdfs"`. Pass the supplied schema as `ont_graph`; start with a fresh data copy for each call.
3. Make a second shape using `sh:targetSubjectsOf dbo:director` instead of the class target. Validate without inference.
4. Explain why “conforms” does not necessarily mean “every intended film record was checked”.
5. Add a selected date to a copy and validate the repaired record with RDFS inference.

**Important distinction:** `sh:targetClass` and `sh:class` already follow relevant `rdfs:subClassOf` paths in the effective data graph. Thus “no extra inference” does not mean “ignore subclasses”. This experiment uses **domain inference** to isolate the additional RDFS step. [SHACL targets](https://www.w3.org/TR/shacl/#targetClass)

</details>


In [ ]:
s4_data = parse_shacl_turtle("""
ex:ImportedFilm dbo:director dbr:Christopher_Nolan .
""")

# An explicit, simplified teaching schema, not an automatic DBpedia import.
s4_schema = parse_shacl_turtle("""
dbo:director a rdf:Property ;
    rdfs:domain dbo:Film .
""")


### S4a — Use a class target without inference

**Question:** Does the imported record meet the target definition?

**Use:** `s4_data`, `s4_schema`; no previous student answer.

**Task:** Build `s4_shapes` and validate without additional inference.


**Predict before running:** Identify the input's type evidence separately from the expected conformance result.

**My prediction and reason:** _Write here._


<details>
<summary>How to — Use a class target without inference</summary>

`ont_graph` supplies schema statements. `inference="none"` requests no additional RDFS expansion. Store a named report. A conforming report does not enumerate intended records that were never selected.

**Library-call pattern and return values**

```python
case_conforms, case_report, case_text = validate(
    copy_graph(data_graph),
    shacl_graph=shapes_graph,
    ont_graph=copy_graph(schema_graph),
    inference="none",
)
```

These are generic names: use the supplied data/schema and your shapes graph. `case_conforms` is a Boolean, `case_report` is an RDF report graph, and `case_text` is its readable rendering. `ont_graph` supplies additional schema knowledge; the inference setting specifies whether to perform additional expansion before checking. Use distinct result names for different configurations so the earlier report remains available.

</details>


In [ ]:
# S4a — complete your code here.
# Keep earlier results under their own names.


**Observe and explain:** Record the actual result, identify the relevant rule or constraint, and explain your comparison with the prediction.

**My observation and explanation:** _Write here._


<details>
<summary>Show solution — S4a</summary>

```python
s4_shapes = parse_shacl_turtle("""
ex:TargetedFilmDateShape a sh:NodeShape ;
    sh:targetClass dbo:Film ;
    sh:property [
        sh:path ex:releaseDateUSA ;
        sh:minCount 1
    ] .
""")
s4_none_conforms, s4_none_report, s4_none_text = validate(
    copy_graph(s4_data), shacl_graph=s4_shapes,
    ont_graph=copy_graph(s4_schema), inference="none"
)
display_rows([{"configuration": "class target / none",
    "explicit Film type in input": (EX.ImportedFilm, RDF.type, DBO.Film) in s4_data,
    "conforms": bool(s4_none_conforms), "findings": len(shacl_result_rows(s4_none_report))}])
print_shacl_results(s4_none_report)
assert s4_none_conforms
```

The record is not recognized as a Film in this configuration and is not selected. True with no findings gives no evidence about its missing date. The explicit-type table column describes input facts, not the set of all checked nodes.

</details>


### S4b — Activate the target through RDFS

**Question:** What does the domain rule change?

**Use:** `s4_shapes`; fresh copies of `s4_data` and `s4_schema`.

**Task:** Run the same shape with RDFS and compare both saved results.


**Predict before running:** Predict the film type and date-check outcome.

**My prediction and reason:** _Write here._


<details>
<summary>How to — Activate the target through RDFS</summary>

`inference="rdfs"` expands the effective data before validation. Preserve `s4_rdfs_report` separately from `s4_none_report`.

</details>


In [ ]:
# S4b — complete your code here.
# Keep earlier results under their own names.


**Observe and explain:** Record the actual result, identify the relevant rule or constraint, and explain your comparison with the prediction.

**My observation and explanation:** _Write here._


<details>
<summary>Show solution — S4b</summary>

```python
s4_rdfs_conforms, s4_rdfs_report, s4_rdfs_text = validate(
    copy_graph(s4_data), shacl_graph=s4_shapes,
    ont_graph=copy_graph(s4_schema), inference="rdfs"
)
display_rows([
    {"configuration": "class / none", "conforms": bool(s4_none_conforms)},
    {"configuration": "class / rdfs", "conforms": bool(s4_rdfs_conforms)},
])
print_shacl_results(s4_rdfs_report)
assert not s4_rdfs_conforms
assert SH.MinCountConstraintComponent in shacl_components(s4_rdfs_report)
```

The domain infers a Film type for the director relationship's subject. The record is now selected and fails the minimum date count. Read ImportedFilm, date path and constraint together. No offending date value exists because the count is zero.

</details>


### S4c — Select subjects of a relationship

**Question:** Can selection work without a Film classification?

**Use:** `s4_data`; a new shape.

**Task:** Create the relationship target, validate without inference, and preserve its report.


**Predict before running:** Predict selection and conformance without inference.

**My prediction and reason:** _Write here._


<details>
<summary>How to — Select subjects of a relationship</summary>

`sh:targetSubjectsOf dbo:director` selects subjects using that predicate. The date requirement can stay unchanged while the target changes.

</details>


In [ ]:
# S4c — complete your code here.
# Keep earlier results under their own names.


**Observe and explain:** Record the actual result, identify the relevant rule or constraint, and explain your comparison with the prediction.

**My observation and explanation:** _Write here._


<details>
<summary>Show solution — S4c</summary>

```python
s4_relationship_shapes = parse_shacl_turtle("""
ex:RelationshipDateShape a sh:NodeShape ;
    sh:targetSubjectsOf dbo:director ;
    sh:property [ sh:path ex:releaseDateUSA ; sh:minCount 1 ] .
""")
s4_relationship_conforms, s4_relationship_report, s4_relationship_text = validate(
    copy_graph(s4_data), shacl_graph=s4_relationship_shapes, inference="none"
)
display_rows([{"configuration": "relationship / none",
               "conforms": bool(s4_relationship_conforms)}])
print_shacl_results(s4_relationship_report)
assert not s4_relationship_conforms
```

The relationship target selects ImportedFilm directly from the existing director triple. Its missing date is reported. Comparison with S4a demonstrates that target coverage is separate from satisfying a requirement.

</details>


### S4d — Repair and compare all configurations

**Question:** Why does the final True mean something different from the first True?

**Use:** `s4_shapes`, baseline data/schema and earlier saved reports.

**Task:** Validate the repair and display the four saved outcomes.


**Predict before running:** Predict what adding the date changes.

**My prediction and reason:** _Write here._


<details>
<summary>How to — Repair and compare all configurations</summary>

Add the chosen date to an independent copy. Keep the shape/inference configuration fixed. S4 asks only for presence; S1 additionally checked datatype and maximum count.

</details>


In [ ]:
# S4d — complete your code here.
# Keep earlier results under their own names.


**Observe and explain:** Record the actual result, identify the relevant rule or constraint, and explain your comparison with the prediction.

**My observation and explanation:** _Write here._


<details>
<summary>Show solution — S4d</summary>

```python
s4_repaired_data = copy_graph(s4_data)
s4_repaired_data.add((EX.ImportedFilm, EX.releaseDateUSA, Literal("2008-07-18", datatype=XSD.date)))
s4_repaired_conforms, s4_repaired_report, s4_repaired_text = validate(
    s4_repaired_data, shacl_graph=s4_shapes,
    ont_graph=copy_graph(s4_schema), inference="rdfs"
)
display_rows([
    {"configuration": "class / none / original", "conforms": bool(s4_none_conforms)},
    {"configuration": "class / rdfs / original", "conforms": bool(s4_rdfs_conforms)},
    {"configuration": "relationship / none / original", "conforms": bool(s4_relationship_conforms)},
    {"configuration": "class / rdfs / repaired", "conforms": bool(s4_repaired_conforms)},
])
print_shacl_results(s4_repaired_report)
assert s4_repaired_conforms
assert not list(s4_data.objects(EX.ImportedFilm, EX.releaseDateUSA))
```

The saved Booleans are True, False, False, True. Initially the record was not selected; finally it was selected and met the presence rule. The chosen date is an exercise fact, and the original baseline remains unchanged.

</details>


<details>
<summary>Complete reference solution — S4</summary>

```python
s4_shapes = parse_shacl_turtle("""
ex:TargetedFilmDateShape a sh:NodeShape ;
    sh:targetClass dbo:Film ;
    sh:property [
        sh:path ex:releaseDateUSA ;
        sh:minCount 1
    ] .
""")

for inference_mode in ("none", "rdfs"):
    conforms, report_graph, report_text = validate(
        copy_graph(s4_data),
        shacl_graph=s4_shapes,
        ont_graph=copy_graph(s4_schema),
        inference=inference_mode,
    )
    print("\nInference:", inference_mode, "| conforms:", conforms)
    print_shacl_results(report_graph)
    assert bool(conforms) == (inference_mode == "none")
    if inference_mode == "rdfs":
        assert SH.MinCountConstraintComponent in shacl_components(report_graph)

# Select subjects of the relationship, even without a film type.
s4_relationship_targets = parse_shacl_turtle("""
ex:RelationshipDateShape a sh:NodeShape ;
    sh:targetSubjectsOf dbo:director ;
    sh:property [
        sh:path ex:releaseDateUSA ;
        sh:minCount 1
    ] .
""")

conforms, report_graph, report_text = validate(
    copy_graph(s4_data),
    shacl_graph=s4_relationship_targets,
    inference="none",
)
print("\nRelationship target without inference:", conforms)
print_shacl_results(report_graph)
assert not conforms

s4_repaired = copy_graph(s4_data)
s4_repaired.add((
    EX.ImportedFilm, EX.releaseDateUSA,
    Literal("2008-07-18", datatype=XSD.date),
))
conforms, report_graph, report_text = validate(
    s4_repaired,
    shacl_graph=s4_shapes,
    ont_graph=copy_graph(s4_schema),
    inference="rdfs",
)
assert conforms
```


Without additional inference, `ex:ImportedFilm` has no film classification and is not selected by the class target. RDFS domain inference makes it a film; the date requirement then fails. The relationship-based target selects it directly.

Conformance is relative to **the supplied data, shapes, targets, and inference configuration**. A missing target is not reported as an absent record.

---

**How to read the code and its output**

The first loop calls `validate(...)` twice, using the same shapes and schema but fresh data copies. Its printed `Inference: ... | conforms: ...` heading identifies the configuration; the following helper output describes that particular report.

| Code step | Selected focus node | Result to read |
|---|---|---|
| Class target with `"none"` | None from this record | `True`, then `No validation results.` |
| Class target with `"rdfs"` | `ex:ImportedFilm` | `False`, with a minimum-count finding on `ex:releaseDateUSA`. |
| `s4_relationship_targets` without inference | `ex:ImportedFilm` | `Relationship target without inference: False`, followed by the date finding. |
| Final `s4_repaired` validation with RDFS | `ex:ImportedFilm` | `conforms` is `True`; its assertion succeeds silently. |

The first `True` does not establish that the record's statements or fields are complete. The class target has not selected this untyped record, so its missing date has not been checked. RDFS derives the film type from the domain of `dbo:director`; this activates the target and exposes the missing value. The relationship target selects the record directly.

For the failing runs, read focus node, date path and `MinCountConstraintComponent` together. There is no offending date value: the count is zero.

**Reading the final state**

The repair adds a chosen date to a copy. S4's shape checks presence only; S1 additionally checked datatype and maximum count. The final `assert conforms` prints nothing. Inspect `conforms` or call `print_shacl_results(report_graph)` at that point to see the repaired result.

These variables now contain the **last repaired validation**, not the preceding failure. The follow-up shows separately that subclass recognition already affects class targets and range inference can satisfy a class constraint.

</details>


### S4 follow-up micro-experiments

Predict before running each experiment. Keep subclass recognition separate from range inference; both have independent data and shapes.


In [ ]:
subclass_data = parse_shacl_turtle("""
ex:SuperheroFilm rdfs:subClassOf dbo:Film .
ex:NewSuperheroFilm a ex:SuperheroFilm .
""")
subclass_shapes = parse_shacl_turtle("""
ex:SubclassDateShape a sh:NodeShape ;
    sh:targetClass dbo:Film ;
    sh:property [ sh:path ex:releaseDateUSA ; sh:minCount 1 ] .
""")


### S4e — Recognize subclass instances

**Question:** Does class-target checking already recognize subclasses?

**Use:** `subclass_data`, `subclass_shapes`.

**Task:** Validate and identify the actual focus node.


**Predict before running:** Will the subclass instance be selected with inference none? Identify the relevant supplied triple.

**My prediction and reason:** _Write here._


<details>
<summary>How to — Recognize subclass instances</summary>

`sh:targetClass` already follows the appropriate `rdf:type/rdfs:subClassOf*` path in the effective data graph. This does not require an additional general RDFS expansion.

</details>


In [ ]:
# S4e — complete your code here.
# Keep earlier results under their own names.


**Observe and explain:** Record the actual result, identify the relevant rule or constraint, and explain your comparison with the prediction.

**My observation and explanation:** _Write here._


<details>
<summary>Show solution — S4e</summary>

```python
s4_subclass_conforms, s4_subclass_report, _ = validate(
    copy_graph(subclass_data), shacl_graph=subclass_shapes, inference="none"
)
display_rows([{"experiment": "subclass / none", "conforms": bool(s4_subclass_conforms)}])
print_shacl_results(s4_subclass_report)
assert not s4_subclass_conforms
assert EX.NewSuperheroFilm in set(s4_subclass_report.objects(None, SH.focusNode))
```

The subclass instance is selected and fails date minimum count. No additional inference does not mean ignoring subclasses. The relevant hierarchy must be available in the effective data graph.

</details>


In [ ]:
range_data = parse_shacl_turtle("""
dbr:Interstellar a dbo:Film ;
    dbo:director dbr:Christopher_Nolan .
""")
range_schema = parse_shacl_turtle("""
dbo:director rdfs:range dbo:Director .
""")
range_shapes = parse_shacl_turtle("""
ex:DirectorTypeOnlyShape a sh:NodeShape ;
    sh:targetClass dbo:Film ;
    sh:property [ sh:path dbo:director ; sh:class dbo:Director ] .
""")


### S4f — Supply inferred class evidence

**Question:** Can range inference satisfy a class check?

**Use:** `range_data`, `range_schema`, `range_shapes`.

**Task:** Run the two configurations independently.


**Predict before running:** Predict both configurations and whether the original input changes.

**My prediction and reason:** _Write here._


<details>
<summary>How to — Supply inferred class evidence</summary>

Supply `ont_graph` and select the inference mode. Keep both reports. Passing `sh:class` after inference does not establish that the type was explicit in the input.

</details>


In [ ]:
# S4f — complete your code here.
# Keep earlier results under their own names.


**Observe and explain:** Record the actual result, identify the relevant rule or constraint, and explain your comparison with the prediction.

**My observation and explanation:** _Write here._


<details>
<summary>Show solution — S4f</summary>

```python
s4_range_none_conforms, s4_range_none_report, _ = validate(
    copy_graph(range_data), shacl_graph=range_shapes,
    ont_graph=copy_graph(range_schema), inference="none"
)
s4_range_rdfs_conforms, s4_range_rdfs_report, _ = validate(
    copy_graph(range_data), shacl_graph=range_shapes,
    ont_graph=copy_graph(range_schema), inference="rdfs"
)
display_rows([
    {"configuration": "range / none", "conforms": bool(s4_range_none_conforms)},
    {"configuration": "range / rdfs", "conforms": bool(s4_range_rdfs_conforms)},
])
print_shacl_results(s4_range_none_report)
print_shacl_results(s4_range_rdfs_report)
assert not s4_range_none_conforms and s4_range_rdfs_conforms
assert (DBR.Christopher_Nolan, RDF.type, DBO.Director) not in range_data
```

The check fails without range inference and succeeds with the inferred director type. The original data still lacks that explicit triple. This supplies evidence for an existing value, not a missing relationship or date.

</details>


<details>
<summary>Original follow-up description and complete reference code</summary>

### S4 follow-up — subclass recognition and inferred director types

The following complete experiments clarify two common interpretations. Run them and explain their outcomes.

- A resource typed as a subclass of `dbo:Film` is selected when the relevant subclass statement is in the effective data graph, even with `inference="none"`.
- A range declaration can infer a director type before a `sh:class` check. Passing that check therefore does not prove that an explicit `rdf:type` statement was present in the original record.

These experiments use their own complete shapes and graphs, so they also run before you finish S4. Referenced sequels or other records from the earlier graph are not silently loaded or treated as complete datasets.


```python
subclass_data = parse_shacl_turtle("""
ex:SuperheroFilm rdfs:subClassOf dbo:Film .
ex:NewSuperheroFilm a ex:SuperheroFilm .
""")
subclass_shapes = parse_shacl_turtle("""
ex:SubclassDateShape a sh:NodeShape ;
    sh:targetClass dbo:Film ;
    sh:property [ sh:path ex:releaseDateUSA ; sh:minCount 1 ] .
""")
subclass_conforms, subclass_report, _ = validate(
    subclass_data, shacl_graph=subclass_shapes, inference="none"
)
print("Subclass instance without date conforms:", subclass_conforms)
assert not subclass_conforms
assert EX.NewSuperheroFilm in set(subclass_report.objects(None, SH.focusNode))

range_data = parse_shacl_turtle("""
dbr:Interstellar a dbo:Film ;
    dbo:director dbr:Christopher_Nolan .
""")
range_schema = parse_shacl_turtle("""
dbo:director rdfs:range dbo:Director .
""")
range_shapes = parse_shacl_turtle("""
ex:DirectorTypeOnlyShape a sh:NodeShape ;
    sh:targetClass dbo:Film ;
    sh:property [ sh:path dbo:director ; sh:class dbo:Director ] .
""")
for inference_mode in ("none", "rdfs"):
    conforms, report_graph, _ = validate(
        copy_graph(range_data),
        shacl_graph=range_shapes,
        ont_graph=copy_graph(range_schema),
        inference=inference_mode,
    )
    print("Director class check, inference =", inference_mode, ":", conforms)
    assert bool(conforms) == (inference_mode == "rdfs")

assert (DBR.Christopher_Nolan, RDF.type, DBO.Director) not in range_data

```

</details>


## S5 extension — Require particular label languages

**What is this about?** A total label count cannot express which language categories must be represented.

**After this extension, you can:** distinguish total and qualified counts, identify the failing nested requirement, and verify a coverage repair.

**Given:** the English-only graph below. **Your result:** a count table and two preserved reports.

**Application requirements:** Preserve the basic profile (at least one label, allowed `de`/`en`/`uk` language ranges, and unique language tags), and additionally require at least one German-qualified label and at least one English-qualified label. Ukrainian remains allowed but optional.


<details>
<summary>Original complete task description — S5 extension</summary>

### Optional extension — require particular label languages

To demand a German **and** an English label, add a separate qualified minimum-count condition for each language. The value shape determines which labels count toward that condition. This extends the “allowed languages” shape rather than changing what `sh:languageIn` means.

**Task:** extend the S5 shape so that the supplied `s5_only_english` graph fails and a graph with both `@de` and `@en` labels passes. Preserve the original allowed-language and uniqueness rules.

</details>


In [ ]:
s5_only_english = parse_shacl_turtle("""
dbr:The_Dark_Knight a dbo:Film ; rdfs:label "The Dark Knight"@en .
""")


### S5d — Count qualifying values

**Question:** What is missing when every present label is acceptable?

**Use:** `s5_only_english`; complete shape is built independently here.

**Task:** Build `s5_required_languages`, validate and display counts plus the report.


**Predict before running:** Record total, English and German counts and predict conformance.

**My prediction and reason:** _Write here._


<details>
<summary>How to — Count qualifying values</summary>

`sh:qualifiedValueShape` defines which values count; `sh:qualifiedMinCount` sets their minimum. Use a separate condition for each required language, retaining allowed-language and uniqueness rules.

**Nested qualified-value syntax**

This neutral example requires at least one French caption for resources of an example class:

```turtle
ex:ExampleShape a sh:NodeShape ;
    sh:targetClass ex:ExampleClass ;
    sh:property [
        sh:path ex:caption ;
        sh:qualifiedValueShape [
            sh:languageIn ("fr")
        ] ;
        sh:qualifiedMinCount 1
    ] .
```

The outer property shape chooses the values using `sh:path`. The nested shape determines which of those values qualify. The qualified minimum applies to that subset. In your exercise, choose the actual property and language categories from its stated application requirements; retain its general admissibility and uniqueness conditions separately.

</details>


In [ ]:
# S5d — complete your code here.
# Keep earlier results under their own names.


**Observe and explain:** Record the actual result, identify the relevant rule or constraint, and explain your comparison with the prediction.

**My observation and explanation:** _Write here._


<details>
<summary>Show solution — S5d</summary>

```python
s5_required_languages = parse_shacl_turtle("""
ex:RequiredLanguageFilmShape a sh:NodeShape ;
    sh:targetClass dbo:Film ;
    sh:property [
        sh:path rdfs:label ;
        sh:minCount 1 ;
        sh:languageIn ("de" "en" "uk") ;
        sh:uniqueLang true
    ] ;
    sh:property [
        sh:path rdfs:label ;
        sh:qualifiedValueShape [ sh:languageIn ("de") ] ;
        sh:qualifiedMinCount 1
    ] ;
    sh:property [
        sh:path rdfs:label ;
        sh:qualifiedValueShape [ sh:languageIn ("en") ] ;
        sh:qualifiedMinCount 1
    ] .
""")
s5_coverage_before_conforms, s5_coverage_before_report, _ = validate(
    copy_graph(s5_only_english), shacl_graph=s5_required_languages, inference="none"
)
s5_before_labels = list(s5_only_english.objects(DBR.The_Dark_Knight, RDFS.label))
display_rows([{"record": "English only", "total": len(s5_before_labels),
    "English": sum(l.language == "en" for l in s5_before_labels),
    "German": sum(l.language == "de" for l in s5_before_labels),
    "conforms": bool(s5_coverage_before_conforms)}], title="Counts for the simple-tag fixtures")
print_shacl_results(s5_coverage_before_report)
assert not s5_coverage_before_conforms
assert SH.QualifiedMinCountConstraintComponent in shacl_components(s5_coverage_before_report)
```

Counts are total 1, English 1, German 0; conformance is False. English itself is valid. The unmet requirement is German-qualified minimum count. The displayed count code concerns the simple en/de tags in these fixtures; the actual constraint uses language-range matching.

</details>


### S5e — Repair language coverage

**Question:** Why is a total count of two insufficient by itself?

**Use:** `s5_only_english`, `s5_required_languages`, saved first report.

**Task:** Validate the changed record and compare both sets of counts.


**Predict before running:** Predict the counts after adding German. Could English plus Ukrainian satisfy a plain minimum of two?

**My prediction and reason:** _Write here._


<details>
<summary>How to — Repair language coverage</summary>

Add a German literal to a copy, retaining English. Preserve the before/after reports under distinct names.

</details>


In [ ]:
# S5e — complete your code here.
# Keep earlier results under their own names.


**Observe and explain:** Record the actual result, identify the relevant rule or constraint, and explain your comparison with the prediction.

**My observation and explanation:** _Write here._


<details>
<summary>Show solution — S5e</summary>

```python
s5_with_german = copy_graph(s5_only_english)
s5_with_german.add((DBR.The_Dark_Knight, RDFS.label, Literal("The Dark Knight", lang="de")))
s5_coverage_after_conforms, s5_coverage_after_report, _ = validate(
    s5_with_german, shacl_graph=s5_required_languages, inference="none"
)
s5_after_labels = list(s5_with_german.objects(DBR.The_Dark_Knight, RDFS.label))
display_rows([
    {"record": "before", "total": len(s5_before_labels),
     "English": sum(l.language == "en" for l in s5_before_labels),
     "German": sum(l.language == "de" for l in s5_before_labels),
     "conforms": bool(s5_coverage_before_conforms)},
    {"record": "after", "total": len(s5_after_labels),
     "English": sum(l.language == "en" for l in s5_after_labels),
     "German": sum(l.language == "de" for l in s5_after_labels),
     "conforms": bool(s5_coverage_after_conforms)},
])
print_shacl_results(s5_coverage_after_report)
assert s5_coverage_after_conforms
```

The new row has one English and one German label and passes. Equal title text is harmless because tags differ. English plus Ukrainian could meet an unqualified minimum of two without German; qualified counts express the required categories. Ukrainian remains optional. The validator does not generate a translation.

</details>


<details>
<summary>Complete reference solution — S5 extension</summary>

```python
s5_required_languages = parse_shacl_turtle("""
ex:RequiredLanguageFilmShape a sh:NodeShape ;
    sh:targetClass dbo:Film ;
    sh:property [
        sh:path rdfs:label ;
        sh:minCount 1 ;
        sh:languageIn ("de" "en" "uk") ;
        sh:uniqueLang true
    ] ;
    sh:property [
        sh:path rdfs:label ;
        sh:qualifiedValueShape [ sh:languageIn ("de") ] ;
        sh:qualifiedMinCount 1
    ] ;
    sh:property [
        sh:path rdfs:label ;
        sh:qualifiedValueShape [ sh:languageIn ("en") ] ;
        sh:qualifiedMinCount 1
    ] .
""")
s5_only_english = parse_shacl_turtle("""
dbr:The_Dark_Knight a dbo:Film ; rdfs:label "The Dark Knight"@en .
""")
conforms, report_graph, _ = validate(
    s5_only_english, shacl_graph=s5_required_languages, inference="none"
)
assert not conforms
assert SH.QualifiedMinCountConstraintComponent in shacl_components(report_graph)

with_german = copy_graph(s5_only_english)
with_german.add((
    DBR.The_Dark_Knight, RDFS.label, Literal("The Dark Knight", lang="de")
))
conforms, report_graph, _ = validate(
    with_german, shacl_graph=s5_required_languages, inference="none"
)
assert conforms
```


The two qualified conditions each count only labels matching their own language range. German and English values must therefore both be present. [SHACL qualified value shapes](https://www.w3.org/TR/shacl/#QualifiedValueShapeConstraintComponent)

---

**How to read the code and its output**

This solution does **not** print its validation results. The first `validate(...)` assigns a Boolean to `conforms` and an RDF report to `report_graph`. `assert not conforms` succeeds silently, meaning that the English-only graph fails. The next assertion silently confirms that `QualifiedMinCountConstraintComponent` occurs in that report.

To inspect this first result, evaluate `conforms` or call `print_shacl_results(report_graph)` **immediately after the first validation**, before the second call replaces the variables. The report identifies the film as focus node, `rdfs:label` as path, and the qualified minimum as the failed condition.

**Why the qualified counts change the result**

| Data graph | Total labels | English-qualified | German-qualified | Conforms? |
|---|---|---|---|---|
| `s5_only_english` | 1 | 1 | 0 | `False` |
| `with_german` | 2 | 1 | 1 | `True` |

The first English literal is valid. It satisfies allowed languages, uniqueness, the general label minimum, and the English qualified minimum. The **German** nested language shape finds zero matching values, which is fewer than its required one. The failure therefore concerns missing coverage, not an invalid English label.

After adding the German value, the second validation sets `conforms` to `True`. Its final assertion succeeds without printing anything. `conforms` and `report_graph` now describe **this repaired graph**; inspecting them at the end no longer shows the first failure.

A plain minimum count of 2 would only require two labels: English and Ukrainian could satisfy it while German remained absent. Qualified counts specify which languages must be represented. Ukrainian stays allowed but optional because no Ukrainian-qualified minimum is present. The validator checks these counts; it does not generate the missing translation.

</details>


## Discussion — RDF, RDFS, and SHACL together

Use the final discussion to explain the following in your own words:

1. Which statements were explicitly asserted, and which were added by RDFS inference?
2. Which application requirements did the shapes impose?
3. Why does a missing date not by itself produce an RDFS validation error?
4. How can a graph conform while an intended record is not selected by any target?
5. Why can inference change a validation result without changing the shapes?
6. What information do a graph drawing and a SHACL validation report each contribute?

Keep the data graph, schema graph, shapes graph, and report graph as separate Python variables. For a new experiment, start with a fresh copy of the supplied baseline. Shapes validate the supplied graph; they do not automatically retrieve external descriptions for every DBpedia IRI.

<details>
<summary>Show discussion answer guide</summary>

1. The asserted union contains the original data and schema triples. The materialized working copy additionally contains their RDFS consequences; compare it with that asserted union to identify newly inferred triples.
2. Our application asks for a selected US date, suitable numeric values, classified director resources, and label rules. These are explicit shape requirements.
3. A missing date supplies no date value to classify. RDFS does not enforce our application's minimum count; SHACL expresses that requirement.
4. A class target selects records recognized as instances of its class. An untyped imported record can therefore escape that target. Relationship targets can select records by a property they use.
5. Inference may introduce a type that activates a target, or a type that satisfies a class constraint. The validation graph has changed even though the shapes have not.
6. The drawing helps inspect statements and their connections. The report identifies a checked focus node, the relevant path, and the failed constraint. A highlighted node alone does not explain which rule failed.

</details>


### Transfer case Q — Which layer needs attention?

**What is this about?** Diagnose the responsible layer before making an edit.

Record Q is explicitly typed as a film and contains exactly one plain-string US date. The application's class-target shape requires exactly one `xsd:date`.

**Predict and justify:** Should the data, RDFS schema, SHACL shape, or target/inference configuration be changed first? Explain why. What would a passing report establish about datatype/cardinality, and what would remain unknown about historical correctness?

**My diagnosis and justification:** _Write here._

You may verify your reasoning with the existing S1 shape; no new library feature is needed.


<details>
<summary>Show answer — Transfer case Q</summary>

The target selects Q and the application profile is explicit. Repair the data's representation using a factually justified typed date. Weakening the shape would change the requirement; excluding Q from the target would avoid checking the intended record.

Passing establishes the selected record's stated datatype/cardinality conditions in that configuration. It does not independently establish historical accuracy.

A Turtle parse exception requires fixing syntax before validation can be performed. It is different from a successfully produced report containing SHACL findings.

</details>
